# Crawl corpus ViBioMIR trên Kaggle

Notebook tự chứa code crawl, không cần clone repo hoặc upload file Python.
Phần chẩn đoán đã tích hợp: Output tự tạo `<run>_diagnostics.zip` để debug.
Bật **Internet** trong cài đặt notebook; dùng CPU, không cần GPU.
Chỉnh ô **1. Cấu hình**, rồi chạy các ô theo thứ tự.

ID bắt đầu và kết thúc đều được tính vào khoảng chọn. Đây là **ID gốc**,
không phải số thứ tự dòng. Giữ nguyên ID/URL nguồn, không dịch hoặc sinh lại text.
Trích HTML/PDF có text; Selenium thử lại URL lỗi, trang JS hoặc text rỗng/QC lỗi và render JavaScript. Chưa có OCR.
Dùng chung `priority_probe.py` với notebook 97 hostname: header trình duyệt,
timeout/retry có ngân sách, giữ lỗi theo từng lần thử và kiểm tra challenge.
Tài liệu nguồn: [ViBioMIR](https://huggingface.co/datasets/AIGuruTinix/ViBioMIR).


## 1. Cấu hình — chỉnh ID tại đây

Mặc định chọn ID **583–1582**. Để thử trước, đặt `MAX_NEW = 10`;
sau đó đổi thành `None` và chạy lại ô crawl để làm tiếp.
Giữ nguyên cấu hình tải trang/browser và phiên bản code khi resume cùng run.
Với khoảng lớn, chia thành nhiều run nhỏ: crawler hiện tại nạp danh sách công việc vào RAM.

`SOURCE_PARQUET` để trống sẽ tải nguồn đã chốt từ Hugging Face.
Nếu đã thêm Parquet vào Kaggle Input, điền đường dẫn `/kaggle/input/.../links_corpus.parquet`.
`RESUME_RUN_PATH` tùy chọn trỏ tới **thư mục run đã giải nén** từ notebook này trong Kaggle Input.
Để nâng cấp run cũ hoặc thử lại lỗi, đặt `RECOVERY_RUN_PATH` và chọn `RUN_TAG` mới.
Recovery dùng lại raw 2xx đã kiểm tra hash, chạy lại extractor/QC, rồi crawl URL còn lỗi.
`RESPECT_ROBOTS = False` tải trực tiếp giống `domain_samples`; đặt `True` để
kiểm tra robots trước các request trang. Trạng thái robots không được chuyển sang Selenium.


In [ ]:
# ===== CHỈNH CẤU HÌNH TẠI ĐÂY =====
ID_FROM = 583
ID_TO = 1582              # Bao gồm ID này
WORKERS = 4
DELAY = 1.0
ATTEMPTS = 3
MAX_NEW = None            # None: toàn bộ còn lại; 10: thêm tối đa 10 URL
USER_AGENT = 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/130.0.0.0 Safari/537.36'
PAGE_CONNECT_TIMEOUT = 20.0
PAGE_READ_TIMEOUT = 45.0
PAGE_REQUEST_BUDGET = 180.0
RESPECT_ROBOTS = False    # False: giống domain_samples; True: kiểm tra robots
ROBOTS_ATTEMPTS = 3
ROBOTS_CONNECT_TIMEOUT = 15.0
ROBOTS_READ_TIMEOUT = 30.0
ROBOTS_RETRY_BUDGET = 90.0
ROBOTS_FAILURE_TTL = 30.0
BROWSER_TEST_MODE = 'failed_only'  # 'off', 'failed_only', 'all'
BROWSER_USER_AGENT = None  # Dùng User-Agent thực của Chrome
BROWSER_LOAD_TIMEOUT = 45.0
BROWSER_WAIT_SECONDS = 15.0
BROWSER_ATTEMPTS = 2
CHROME_BINARY = None      # Selenium Manager tự tìm/cài Chrome; hoặc đường dẫn có sẵn

SOURCE_PARQUET = None     # None: tự tải; hoặc '/kaggle/input/.../links_corpus.parquet'
RESUME_RUN_PATH = None    # None: mới/tiếp tục session; hoặc thư mục run trong Kaggle Input
RECOVERY_RUN_PATH = None  # Run cũ để reuse raw thành công, thử lại URL lỗi trong run mới
RUN_TAG = None            # None: ids_<ID_FROM>_<ID_TO>_synced; đổi khi muốn run mới

from pathlib import Path
import json
import shutil
import sys

if sys.version_info < (3, 11):
    raise RuntimeError('Code crawl yêu cầu Python 3.11 trở lên.')
WORKDIR = Path('/kaggle/working')
DATASET_REVISION = '0148f6f80ffafed5c005af6d506ccfd9d3fb47a7'
SOURCE_SHA256 = 'dee1153f5b9d9010dea18e1b7724b4915ff847b711d036bec8b933486802a971'
SOURCE_URL = f'https://huggingface.co/datasets/AIGuruTinix/ViBioMIR/resolve/{DATASET_REVISION}/links_corpus.parquet'

if type(ID_FROM) is not int or type(ID_TO) is not int or ID_FROM < 0 or ID_TO < ID_FROM:
    raise ValueError('ID_FROM và ID_TO phải là số nguyên; ID_TO >= ID_FROM >= 0.')
if type(WORKERS) is not int or WORKERS < 1 or type(ATTEMPTS) is not int or ATTEMPTS < 1:
    raise ValueError('WORKERS và ATTEMPTS phải là số nguyên dương.')
if not isinstance(DELAY, (int, float)) or not 0 <= DELAY < float('inf'):
    raise ValueError('DELAY phải hữu hạn và không âm.')
if MAX_NEW is not None and (type(MAX_NEW) is not int or MAX_NEW < 1):
    raise ValueError('MAX_NEW phải là None hoặc số nguyên dương.')
if RECOVERY_RUN_PATH and RESUME_RUN_PATH:
    raise ValueError('Chỉ chọn RECOVERY_RUN_PATH hoặc RESUME_RUN_PATH.')
tag = RUN_TAG or f'ids_{ID_FROM}_{ID_TO}_synced' + ('_recovery' if RECOVERY_RUN_PATH else '')
if not isinstance(tag, str) or tag in ('.', '..') or any(separator in tag for separator in ('/', chr(92))):
    raise ValueError('RUN_TAG chỉ là tên thư mục, không phải đường dẫn.')
RUN_DIR = WORKDIR / 'runs' / tag
SOURCE_PATH = WORKDIR / 'data' / 'links_corpus.parquet'
RUN_DIR.parent.mkdir(parents=True, exist_ok=True)
if RESUME_RUN_PATH:
    previous = Path(RESUME_RUN_PATH)
    if not (previous / 'manifest.json').is_file():
        raise ValueError('RESUME_RUN_PATH phải chứa manifest.json, records/ và raw/.')
    if not RUN_DIR.exists():
        shutil.copytree(previous, RUN_DIR)
    else:
        print('Đã có output trong session; dùng run hiện tại, không copy đè.')
if (RUN_DIR / 'manifest.json').exists():
    if (RUN_DIR / 'crawl_config.json').exists() and not (RUN_DIR / 'corpus_config.json').exists():
        raise ValueError('Run cũ cần nâng cấp: chọn RUN_TAG mới và dùng RECOVERY_RUN_PATH.')
    saved = json.loads((RUN_DIR / 'manifest.json').read_text(encoding='utf-8'))
    if saved.get('id_start_inclusive') != ID_FROM or saved.get('id_stop_exclusive') != ID_TO + 1:
        raise ValueError('Khoảng ID không khớp run đã có. Chọn RUN_TAG mới.')
print(f'ID: {ID_FROM} đến {ID_TO}, bao gồm cả hai đầu')
print(f'Output: {RUN_DIR}')


## 2. Cài dependencies

Lượt mới dùng phiên bản tương thích mà pip chọn trong môi trường Kaggle,
Phiên bản thực tế được lưu cùng run.
Khi khôi phục run, notebook cài lại dependencies trực tiếp đã ghi nhận.
Nếu bật Selenium, bước crawl tự kiểm tra thư viện Chrome trên Linux, cài phần thiếu
bằng apt và mở trang thử trước khi tải URL. Kaggle cần Internet để cài package/driver.


In [ ]:
import subprocess
import importlib.metadata

package_names = ['pyarrow', 'requests', 'trafilatura', 'pymupdf', 'tld']
if BROWSER_TEST_MODE != 'off':
    package_names.append('selenium')
requirements_path = RUN_DIR / 'notebook_requirements.txt'
if requirements_path.exists():
    install_args = ['-r', str(requirements_path)]
else:
    install_args = package_names
RUN_DIR.mkdir(parents=True, exist_ok=True)
(RUN_DIR / 'logs').mkdir(exist_ok=True)
installation = subprocess.run(
    [sys.executable, '-m', 'pip', 'install', '--quiet', *install_args],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, encoding='utf-8', errors='replace',
)
with (RUN_DIR / 'logs' / 'pip_install.log').open('a', encoding='utf-8') as log:
    log.write(installation.stdout or '')
if installation.stdout:
    print(installation.stdout)
if installation.returncode:
    raise subprocess.CalledProcessError(installation.returncode, installation.args, output=installation.stdout)
versions = {name: importlib.metadata.version(name) for name in package_names}
print(json.dumps(versions, indent=2))


## 3. Chuẩn bị code crawl

Ô dưới giải nén snapshot nhúng trong notebook vào output; không tải code từ mạng.
Các file `.py` sinh ra có thể mở để xem. Snapshot có sẵn của run được giữ nguyên khi resume.
Snapshot code được lưu trực tiếp trong notebook này.


In [ ]:
import base64
import hashlib
import zlib

EMBEDDED_FILES = {'audit_snapshot.py': 'c-pO6TW{RP6@EA1|1ep%AldFxvf~)7v@6509mGhS7>-jEP!`OH9PTpW%|lYsMp~dn9{N<EK?)Q>5g<-b1Zhw-Mf+qu^g%)Xhx$wUotYt*yV_mZXaiWw%el?Dd~@b7olfW7FpQ&AuqZ9XNJiyk!_RP;mMn<)D9ws8@(ae(0RLE(#(Yo2u?UzH2az~r#aM`vg_&fW`H~;TOjfDe>2$7MiIO~%CF63Gb6JS1S3;R3%+J8?mr<4$%uw9TDrl4q$LQlIZ*IoC7{}4x?7mE5#tZf9iNfc(Xx@EPWT{b;^OD$V6_0S+kV&j2OJ=t-0mY<fuB$Zivp@vA<X5j4sqqn%w+v~B^9h$SJ9M$qrxhFm1M{cEcJ)diLa5+@O(r>8SFbSqCq-GwR2l5X84n8e)+IWx6i1~kQYeqo(ZH(8u=fkgac0u|EXlcF+C;b@&|;SRQ5L0k%X&A>WWwX<V_{qGym8m+GR5c?c^s8?TW~SvHgPNnH6d{E`o?RwG=)1BM?o|a1thfA$qzjhm{JO1?`6TH%j%WXLIa@;V*^I-U47fEx63Rk@|gR=ax{GuvNS7eHn%8cl-sqimX+^hsQ|Y~@^a#lAwlL>2@=`Tq@|o(WFli-3N49EN4C-8bd@+wE{=SWm&{hQf>!1}qJ)<B5pvmOzv6Kv)b(nss#Mhg-;2efZME+bj<D4d6t#qp>gsXt$dTgEs$pp1!x2<R1CPXcp3`P^c@TKq@Yq(bmsKSqFU8X;k|G#9k(F4_pbNZLAa5vTCb<{@Bf1Dg$g8*<T;JR@0>D+kR3k_Ikfg#k@1Yv6sRV6$?Bd?MNq8EC$RWH|B|?-N%&H%|R3EK8p!}?93ZCS#P!zG{=*roflmc2E_R63Uy=y2voxRqu9U~qUq7^^edQZ`+jAuzyB)s&;wE<cVOx<mjZQ&^J1D2(1ooUO9TGt@qFU%g+>6QZ@Re>DKT`LM;GQ&kBV{53CNm8bjOpR$-C4Dk2e`?fGrr9kYC<1HPWnmNpYm33IO`?&SQHsBDghH-0JI>H?%goCs>ha9~Ywb}QrsVos&Wm_D$V8#)Te-Sk$RiD-wdW<OQU;V`yBbNg({rs?LRFI%h7_(H#o3;1{mkT0?Ootz#p6qSYlKXjg&VHc+8AhKk=lk-r5xQiK3u!_$5jeFgyXR&1<9gK7XVOT+i6rhxL0hmLmrv^HIA7-7XE$?d@qaIH$E?f^31TE3Le!iJw7&zSX^7nJd!|^<PPNxatmP=oHFwc9z2B)&>(n4$;+ysckW04D;AyuJgqS1Bcj+HPG^eXNW(!8NE@Ejb`$+aTLNsM4nXY;NiIcE)X{}44d-y7=I7-Yupn_NG=_5t2CNKH*m=wk=aFEm$(m{BqC$&m+)_SP{Sp}fa`klRs@fL@oXq2wwC4#}UdA9QE49ZWFtRKTBLMK8_0vc9-soL$yh5JTCm_}#g7;&Nox-a@%W^1B4RT#%rhT!i_W+NgPm-2*1I|Te7(3%!Vp*N0yC$)QF3)q3QcqQLIm@})<*;jfHH2-JP0^L-r+-RE8~fw4f2AXB0{h|9AHP38{n}^u-hH^ujzrR`I(BgOB`u4M&QCw}U1y0OL|lj!D}V<gU8_RXBQ@R1Z*BxghRdS^IGHlZwBS7vL}ey>Il6F_Shmo?L1>D?1jK4Ox#4oPuv=y?rxqQL%>M_Kek^cW7&M)#$thI|UE+7vnA+lN+F9YMdNBst&TJuib*lvGq%ldLNIidyFk6%ZgxBiXT4v2fI3(nO2o$Hb6WaAc;XD0=g5T;Km!!F-BY4nPRHL<=6=}0F`Hg81;e}NkPg+JgD-F;|xJDJABJx$Aqe*u+hnLx9{HvM{#0aOk#=l0IdqkjZTkgh&qFR{&7lW4Y(d9y;WVBW!TC9Cl0$gX}Sc=ec%()f*uO;e=;JO~>XfJgFFH7!^seAUYVd@Oqt(1z2RA>`u4-_s=0@5Gb()z%)^B_Dy|9*7)<R~tVZaenvt)HCi-o7?;P%0I*?Cyhg+98pAqB6)~eRln|O_g7(y|^a$YjRCVP*rFS5rvoe($mKTYmM)(u?J}&ND56A_TtRn$D%?Zu@?Z5=v+;Z3b^gER2<;gGZLwgoLV6Dwj2vnXr$g<tx17G+>YlgL-cX!gs#d-Ea)N%B~R29#vJPP(rs}b-Q7TDx!%#%ta;_O^FeA5t*lS7%Ib143DtQ)Wnn|OLfx7TXZ;5FmfrRJvwyNe&`bt)yng=t8QSoNJlbaa(fR2o@V?};EstW6vAPMH+W^VpS^F@9`uYB^mHQ7PAIP1B<pY}jq@uikl*gGA@+2v;o{kb-mHM*($_Km1p(s4*s|PzLeN1C^%s9i;$$}%6r*amC5nPf<FX3{(%01-P{-UAFTU%=0Ebh}l04Tdw8;l}t3xf39E*j%@Jm)y!M02Pqz&ZD7k*BEFL^t(;fQp>3&WzRUP-U2}wi$I;pxaR0&IFsoLzTcK6LlKUIYFyb_@d97)Z5p`mikS71i~wmYzeGF$LOJ!qcBp0G;io$$Wqmm1FSTzP-_cRS>$5YY!d0!2a6y=d!U!<ERkE!R|=21UcK{!3+uCe9gxLs9kr$pqO^T4c<e<rw8UMmi`Mq?dfHoUF|TFGF5G3C4%=ayFQBLJvs^%+b*Cz`M1e1YvjJ3Y4}!Oqi|B+(bJ{Epa1DS^^IT?f+_jd7A6FPgn9ag#XV9iOAl3(XUZLmpOd^`YyL*ViwBCGdHG2uFB**MU6Vaifs3c}ig&#$OH#h*Xi?z~rGq~Z*CXmh8Wtm1@PW8;DrO+~f-BK)<sdU)S6}-=m+Q!$b^w%&lt9M&oIH#|}qCp#rpl4r91~E_e0swY%kZ(pP%JY~}7wTk26L9yj`mt}`KtHd_+pfiRal}mZw9atS9I&gqFGJ(hXr#XZ3|KRQQ4E+58h6gkaTPJZ5tF52Zj1|uC#pred=OD$W+rCy2YpNU(@Y1pE>0S{%l1fsnS`fH(2Pi;8Q0bcin(0jZF-o6-Dtg}Rz%J07T+z^kl+yK^VmzPWDkS4IgBt8wa0{<J%OiGd0W=Bof1t9Tm7JR=>H{QRR~rmEcu86Wc6PGD$zt(&ChrG>YGf=!O%H~Iy?PDl=vD%=PXeUI!Xqfsxa-`p=!e>CG)E2Z|G->fMJ;(o<;`8A<leT*Kg%V^mPUX_Nr(bWj~72ts6J<qiu>rZ!DtG7-a6PX#nL>TpaJAe1zP(PK;R`1?-yd``bCj+*sMx^&4O{V(1z;gvoUzkAnzDTRz#sx3FNF{`8;|PfF3lr>iPSi>>RSWcar|;<++mijW>3j^Vvtfn%%K!hP>h@_akPUKYmLp}O1R>Ev*Xws%`E>1KOLQ<rwOW2ngF*0o>WdUe{@jp`ETjIZxJ8b{~P|6R3vG}Id_&%TUs&^SN+UDd!#e*^UWeCOf$^Y1E_j`YtsI{PbzrvmVE^M=dro}YeRLd@AS@MFfInz{1)^gC+H(e#~tm9jk&j(X}cUs31d^XFe{ave^W{qTpguO^86kKgn9^*-z@u{GcZFV3I;B_Z~2I4q$#%NG#_E@#iuF&jl^&)oh-zGDq9j4YeHZ{Y<#fFK!Lp8?(aYhX=5`C2zfVW)2rW2bNSjh$w8#{To0KeA)6+jEGbClwS3!vHU8Dg#CIF5l^6fWp*xWMEMWS<s-raH29^-6032qw4(hGYnH^p6nVop{@QbVP@juAXu+LjTBl@yLnfZ%|P+jBAK=2+Du}Dv9<2D5qDL9FwD_<Q28@U7L{Hc)?Admfi=@#<I+wGA(}QDVhJ4<x7*hCK<^he+va>kqE<su5-1?L#@?D0KxV#V>ts}><8w!+hdKJj`;Xsd`{VP|Kj(EUFgk5FuhWg=R+l?<J9AZM^%#u{59-QjW`wQ!d+*IVJq&Xfx*yVi-#C2);=h9z>5tDh6@~8eoGIYYxWSH1rtMzVUJhaJT3X6#Rj_otGf`hy6q)vVO|pmj#9<Je`1|IQhi|w1v`W!@FFKqqkzeDK>aiM^H|6Y1mqOOIsSPr$fe8&B&_QvX3(noMXFlNO?AyE@kM!)z2|J=!DLDU@vRnd=Q%*w%3M3ewVVq8yQ<f-1D^N=;#!Q$_zvgs|r1=)VXotoWy64|aOaxP~L@cQA63+VvepEl40|$`0g=<|j^n`Hs3wFZZBSr0%pAhv4>-Bm}{oTToVV&yrQ8EO@F`B$7XpU8bk1M@><7ro){950{^zq15ySm+UYcqMfR3HjSKD^Kaf2daZiSp!wd)w^8g|e;R)8F6w4Zs~Vn*mV%Il+25=G8XHu+HD6ZLBBSY)-CLtQ|L8K#e=^-+S=Hdwl=VyN{o^Nw5kvl3-2yf-n3paDGr<#?AlJfS%R!QVfYaPj%a#M_=hZuRi%>`^I!8fQ>EzNg75Y2(nj!mioef^~!$%u6I`r',
 'corpus_runner_snapshot.py': 'c-o~K%WmYx@jhSCkr9xy9#PW1;>?-{MJwA2uU8vAya*-(nj(8<XmLoIO-ie1=O2<okZTMhFp$7VkmK$lhc)~k`jb@kgY4#GB-vVEHPc;PU0qdO&(_Aq#$lO|VnGR8_&E(pm@fS&Az7ZDQbLn?lu$<eBqUjR8b@qFo;}^na>{6a9wl=Uq<K~{5-ezNmZecrFlS?9V-PK~G%tvs&oe(~bTG@)CBVip4T>mD7?A*bX_*u>ACZvG{4y@WC@9q8fSs$4;#VA<I%N^X)o7|dea+HDeO~&-LhP5pCs?EQdWjSAGrMH!W3kAoAHrD%gD>|FkDeX8ATF`Qxi<4MNoc-#{=gbM-T!3w<oWRt%w7!${8%M0!Sm-dDXcNEzDWNR#j(Hj$k`_Lt0)Q63r1cXlbvm6`w4-8#}A*7*N-0#$!?a#^c6k*JSw&xJ@}3D;4!g3|LpjS=OYqFXOw&jyiA8=Zvls(TRRT`0sbRL{>;xKxx^X?B0?Jbm$3hiGa{B>6m*#t48A{ruSK+^X^GTL?r+2V!$-~(hEL0IPH}YSBN!EEK%yDQ#sjgZeElbW%&3~=gI=0(*}>$M8ft7PM}ri&2FU9DcOJ=&I)2nD6oa<p0a9Gg75SxC5&m0!ZstdEnNzPQVzF!`$v(|N1R$bi8d8)H0@Bb+llama)zgL;U!)0zxq$QDJB<=Qzf{8});KlD1>R~nFZ`(R7zIXzOdh|U)R=aEG*}G=1JHiLkI(!i=mxWc+`owNUI6mpm~XuFlW0a+0s1Fr9Zmx3z71exi&zTA!9i=t*G01meg^0F7Cu`jc(EOaj;$g2m0RsHk#ioHh5L^lThnnRzTA%(C13h+N%!+S&28(5gA&9HmcUQuG<4+d7g4bw-P<{7MiZMRL5jD}waQ|)`4LdzGr|hs#<Jd0vIUMl!NvUG%#Oh}4aqFc34S7AwZu9+GLAU#HOlPaRPiB({jTzf6@}K6ugm!&fqcc!k05ejxbDbL%Pfupzo2qCm_??}49Z&O6<%GuIcP*48qysBA#sRb1iai8Z5JF@Fit?=hm9)%CC;$otlUt?OVWa`bLi`c70lKTAV@+rmLwt}ovMTwD$TJ4paa*=#w~{<qK}k0$dU8S*6C7bnQ(yVI!4Z{y~=d}54TPZpOa<8z=#J6uro-^%4Dn9UvgDCr50nk`;z=M_?pviN^yz$1!74cCa0NIAyIr$Xd=dW+Rtct=hyG23`6&Jeh^T`M$#FN;D)7)CV-J2(h<`hU<$xw*^;u6+IeWo8y&1IFz7Q*nT4RtwnWAXzbKh&?(B+XLh@qBQgg#D>t3-@qLbeR0x>2ON`ftlBI|_JCam>4QB*4L47{(2p&mDO<ij11H|!#K=Am-BvyG;<Y<1nyN^_)GbdAvkpj1J!L4IwrOhM@!GeB?Tn5LVO3&wiI1!AhEI&wo{J5SuVE2CgzqjCv;JC0BN;0zr<EO3Q7E7XlyrPsrXkwWo^3&EN~Y4vecvdE&Zi`)+i8iL}2Z!W;+d&Ol&U8{hptwphnpP&bWkhO4yKk7S%lDerqY83EKrl@H=L}|c?5UW=#Kx_^qLYUCHF><_rEd+xZ0~i^4xH-hvp=~O-2?CUE56Oqb6n_WIh&+(YS3ct+j|ytUAIgCBx+>*9`YpOZD(z5CF+^Wy)|Hpe(yWYqkz1?@kXTyg0rdb6#zHbonpj}i3W!=E7h#r&7C-^YY#pZRq9UNULJ}SKn>RvLP8wteF7aw5I0E2cQxT3wJ5wVwf`|uh00+eAMMfTV2-8&mo4j0?x6xt=0WdaH4R1}9l(gO{y}*p<j9!lQ&;Y)R&<s8xg|>kORRSo$=m36+loLN0?gs}dH)GXfx@|M>f%Q%4$5C;qAy6Z;*q;lG+_I?;!X@Y~zZ-Fvk203^V3WI<$V?lrry>pPhJm(*tCoP$i%ApPlqc(a#^mM{!!+y5-2?CG$-$n*!MIwQ0)~}6zNkx?tx>~^@VSZP@^q-H$;1MzJ^*+icX8W*C|AF_Ty>P$kZLhS<D=tYj-Ns10A?+{Fm(3jLz_nx*KE?<Z`#7oH)UvvK%3pzU!I0O$;ZUr%sUSSR!%2hW*pLbW_gr?jf+G&bEV+MvhsrWp@nhWVv+*h?sf$#*98>JixLm&7=Q$mn>o0ItY?v62$F)vM%HHRyd6Rj%$+7y#pEiN64*6Ot$z?kQO&^+UhL?A$zL*+sMQSm**ur5X614T`8(Pt9wTtiKQ?ka%=J_JSWls}aU69364S;4vZ(<zk14e;I*qa50NdMnTO(cL8dj|9!KjB?=QQV3O2pDQPR$rd-2Y#?t<WF2SFJaET~m7Jn9hOtkZDrygdmvYT`f_m8T}E2qFP8!_{xzI<W1p6SwcaECf+Q1Eh$h4Gzx_KjA$J43)tt1+A8+T{nH&hROL!4f7S9(eO{@#<t=^G&-P2xDj-SxB}8Y)In<6x9S(cvK(<s+2kN@ks7LSAk8PbT(q!(P`>7X}X&@I(a#L}+)xW{!PlsCVN*0J*s2w{;P#sF<&kpQ_USPS%wXE8maOJ|Z9X58Jf;oJ}2W)|0BoqkG%Uq#L&80#;7Z85Is_scVfsNd%r<HE|MY@av@AR^OizT#MFjwr47Mb5V8{$YTpix_t&o)^!x1{xpdFdWi`~$N0{@Z|D&{GE6lHmP+6B1v)`&&ecyu2jA_1hmZlxtIWte^f}k|KKlPu?L&Npbz|uTb;H8clHh?t7nL+>CSq9*`E1cLwZxbp7_bB?+(JexEp9m}eV5{`viPfc5&_59F#csH?3j-PW#ft0(0A{ohma<6nOI&-J_S0&*pd!9+K<)72AsqIz1IFi1U`xu|tq9lOR0yFlu9{q`RPiTMy52zSW9Gmsc27k)&}uHXKzAm5a5Jdz`0=1|k8OY-}Z-RIAa|KJ_&zdSfRc9vo1sjWBUsc5gfA;&c@CEH*SzbU~WL&0ss=yN`DDDM(^(}8w#bCdA@$Hp&!t#2nfS;AX8$`;o5g6oQkduAJN$Q44@4s}K9p|<bUW;@~zG0p3?VY$qhEg)sEF~t%R!}`6?f+%u%-vK25W<%HF4^fSgrF#qDO~j?}+41qqk!08M`t7&CFO=btv@=*ILyi5H;PRM!)ix<#2{BCvH{3nz)&B3i{lmk9!y}L_uK8nPIbTE3k{6*_;})tLs0zVCBUK@|Ou7?VNUdhkd_?Yz$VHl;(VV#tM^KP|?SX-EMPoun0pLG)BFjf*28@)tyvc<`OIOKJV*GROp6Xx02KU7Hs_GB$uM@LjGUX`-rfya`n^~h)|6~^BkTp|3SV(8<LFA#jG2Bl;3CCJnML$W-lJp`W`o>P(x4_bjt%z%MsHQ_Kz9NaIX&SHF#9T<0CDd)9w~5QtUV5V1-Qel#?P$l7Ng4JZoruk%1oT`{56GuHYSzA!k6p6eNWR+Tam=h&7108#=*OmD%%k&Oezm$yHOH{lUenS6Nj@e!>sZ=cQPJmDWZ@=s4%bPNSGREb8(rcYAynlcAmULw=sJU(2jWQCo5e_r7EpqiV{j?2%%I&;<b7M!yc&0Oz&wwC_Bvt^OG2+(u140O;g@?I5>fH2P^{{XnSzCLfF1^CcHIZ8@3m(%iDb*#6LTW))pb)_)6pPmbPDTom$r;IdX#v^t@F0kWKTX0fLpyKO_6L5J|WO`1#JP#Mh;J2c*nb+a-}}p-#Y+FgCm8(m%GQGIa*9cW395rDK!qOmDG4kkQH{bE;)4Oae8W7_vDJ<4cE_lPUn7bi8gzDTZ&1Mk}{j;er?KnvKEcCql&NX6(_qGhRVCDHLIsB$isu*9UObRpYFr6gYG2`gSI6k^IKrhc7~JfsZl>Mor}pEp{tByRk-AV*Q|^^72r2&i}H52{FN&IdSR5i?KWGFs4FCm_I8gNe*mEpxLg!gl{E863U3yo#C;=-{UIb`1bVAPP=}JsDUZ*Vy!H_~CJTJV2Kp|UAAuG1lQe-=aEwRK0KGrM%)w8%1<do3znTE!SX@vlp4!OPMg3@|eqa;u8c)6N4Ru~l*6n+s15KLq9CIc#E}$#bBN*p7S{qFXRdED>w(B$sNDs@Hk79uUok}gOhE!!Uz-))rca@(|JDgF{lhKcITxPGeNy!qzvFdvjISmKZ0<&Yx$&?N5h6;g1fiWP`>iBo&xZdZel0P)<2J72av}QQOygRLFZpanHugU#KSl8JW-n^2OTeD@{nPEc;kQBBSl3&$TFt1od7`vaRLKnH4m2>EQ6=tjrsn07Y{mkN$EnLW~sK#-4wbS-7-O(l{?HFjbDK;B}@c3*NNApF|usOAvbDKNp7O=PE(E2gn09826mqB+xr-zd*3XNEEYHE$VM^(dLc)rT=wl%2R6sw#<{>H`<yd0}25%0(#8&fe|HB*O$?Zb93-C#Cp9=JLqG!FP{u9_9%aaYGCDg~0i@owo6z1x=|sP?ORxK;z2@7X%GT<fNQvx)rTHp~g+qbc^R%S0x!gRX{q9SbY3inVF4F>M=O6gD39T%BRx4Qn!_eYGG%OE0+bY`=?AdZ(IA#v=qwn}-f>Ni#HSs;%u=!*w@;*QU+a40W0bHU~<fym-J%vI+BNP!<S7G?!LKQ=!ez39J1{a~94aoe<8-lTVwxYpm)BbA_a#KHx<<<DmW7;$53$*;MfAcR_1NzKv~>(rGrFZ|8icgPT1=ClOZr6^HH^c+uZHX-}=!63;tYO{ZHu1`#rt$kR=Sx3<MtF3HUYqMikhAiD3@UC>sdY#zVk{HxB=m+48neW7px61Ck)pXmh}iuDV<<?TG<T|8d*K6<64g}^Aobos4zS)+TG9zR&y_`3OctH5tqOAX33pt+3@=oTUU&kgJKWKh1HQBUxl7uc-G{D7jVaIv0XuB)4+cYcC1=(Qda=^Yfkvu4Vb1ui3)7hCi}cs)We4SgJTbd0|daB$SaKLZfel~_iz`f~BCnqzIR7vlm0;<E8SS<1R(a+*yL9X?_Z-z$1}0R@CWUjr4AaPiBi49ivGkeGPd)$`w6c!x@SyZIb)+N5~NXY7!&03J8vkz5bzuL0DLw3QBcP0yE0Yz*!pcjm9&7HI@>btj^LqJoFcRhyfl@+tCV8fud2fftwPPD^As3Tya7JkX!&$&>qV?Mes-Hz6F<R;~qCRfJ8E?8`U>Ab1=m*6?4rirMPrM_N_Yt2!Nm0aCqb05H0a*0s2t@?K|J*R0SAC-A&(krqyNX{kpc_~$m=NcfzFMfPtlJy~CDA?xZe;F%oJW@wf8_~|%+kRLYkRkpx~Rmw1EM&x`}3(L*s1fOjTLT)&Kkj3-x;&>i^mEd{2d*I2`53d(1LLineMvB06wEs{*{#68Cn!*18|3UY4',
 'crawler_snapshot.py': 'c-qZ<Ym*#TmEiaK6=^xVStVCjSGOdu-BKB3*?_{b?2%=5Ct5|4U6s|9?yAaiW>!CF^#_Jv23WiS3q!!}OpJ|}m<bp#GsCij#V#M@@c&Q*vdyRc3wvJo=DnFWtE+|B9S5|sGVeY2b<Vlxb<aI_?%WsS7vkiV>$ki?8Xip6lG)_Qo5zzR?|3&8@2y3Y9(j{+Cz@n!Z<-A0-#iWDSv=eIlGKZ5!{`v6JLk@wYmCxl>IK1QkuTCH2)uYYPtx2AXR{;^^EjDhjfQ%c?#{zBi@44pnM|Sql~9$gC5u@erEPB*jl#twAI5`R6`l<iX&TM)4!5N1+#aV<ID9QhCNCXEgGHV|^)L$t$#g!6@@S~*=h0z4iFZ^zc{2@X;ck?2sbQE$c|46&8TA{64}ZUs%%tnlG>j*mMIKLdH)vp3KFE_m)l&oC{YpGn&*Lx~LpRh<{s&%krcoY3#ZZ5lhr=NZG*d6$%#xXUo`(6D`!a_oV_2`@2YF7z*)W;vU%ETl7`v{2C4;>v*S~PlvOFH>X0s#xnA2Rt%Ig=#AQ!WE0LVf^pq?$#366_qMs*L~y_v+bHa=wYNt|0n(_|;fseGy`+`?C%izm@*^jhQ6{3uM*<N%f$W}x)4gNwh&Bl<Wfe+2k|;yHbMtC$>km*vGs({L0|LclZ`Og<T^F6~BnFq#Y-jW51*>-Foe-t>B&zyFNic;hNQDdM^nWl@+8#@CR>qIA96+3*{;UVY)!+rib(zjX8VlCplI(EtkY7Wtsr+G=?4Um8JIW*Q+ov*e(ua1s!4(CWllGD=dwi)Krz593`JQWF=%h9W4~$;RR4rDwQ6XB-_$1yvs=APTT-6LCyH9K`vUkc?k;l6f?1`st3}!U-L5AxB#TOd1%qJ)hh5UG0W>GK~kp&QTs^)Joek#(*zi?ustxx45#%Af|i6IBoKetam$I0Q~~L%JLxDqn}(QkEU|~L^W|#!7Q9c1hRuq-qYU5?;LBC@8Gh>Lx9<Aopkc)+%Ld%4$>GXpL<MwEk32ud=d^K=_`+riz7YD_>!GBWq6LkW4X?7F`Z{kC<kMiWyos7Y!Ju2&xMl=#)m9<miIPWooI$s*z_0qXzf|QrJ9^1;jn1Y#s_Ku3uH}WWpUgAvdAWZgemBifnvpp^b$xbs~>~!caZLz##mC<NLn!M{+*fc0WN#E(=gn+QH>M88(<#xc$$j2(5DU-q7?;@K49D=Ifzn?Z8IdAN&J<liDMN&_z1XRu(VE5&cFG&Ykr%{2uQWa{+$#$-=-m0Z`C^!dKz`I#ZEKzx9{ZMokh1BcKc6bK~JE!%t7hWcV@>I+9wQ9H^#Gj*vaxVo;O=GeTO)G2&co7K^<tdR2don9C`_)(uB=ftzWXp6@}s34*ce>#3QeXAJ;t}_RV+%RNKK7g|QNWWr|%<zrz^#{ujf^B6<lJZ`$-<UF5))y>No4b%d<J3q1wsGK_G68wE}3V4NiIBlKp`fw#5>jdo=6vSg7CBCCJ=N*KWiY5J|sYs}?w=V+SWVF>e4CmW1m4-a@lK-|c@Cwktt9}jUp0c-p|_5ws0;18YrXdY$2V1pP$1`rmSWeB4<nhcv*yG4bU&caKFa}awX*iIOl7Y{*N8hW@aAYVb_C>kxMvrG*+9%ei!CdqI|kEyMS_Nedu7XPD03KcuE#bg4~5C~Q{qWPdh&%ladlb;RpCX{J)Kwz350ouSMOn{%`yJbUoCC)OD1&0g3tpRLHYGJOwsQXPVVb?{jYx3(+oB}Vd`S_&w#;qG(xX8x|Fv%k?&b(|f8pVSc_#2`1JWXITf#xrJ`#_MmC3+(m<dL@%!FTu(<%2PDY;Fyz?1b<JsR0%aIBy!z3t&#g?9rMVgT`vO6i|r@cerS?3oD$!WKCk+v5N9C<A^aa%ngW9GoXBc(8h&rDQp5No+3D)<1dZ#e4c$~YkhtDOY21V1tT;6t4#mMzdoapt0{*W+wzW~_7mxeOuhOK<Uf$xk(H$r#Rk@)sSB;`&A#S@W({7KZLZk&p6+=Y<w0hPY13T9&OUYyh`;4s@iw~MZgmi5@hWV9$%G8-)q9@cS0KX#7G<EBW%#M*CqyN{!zv0VpY&dOYY|S?a8m$IG)V@eq_A9*Mo1QEytANphyz$I<F6xNMYtO=UcI#d{RSo$b(9W@h<njd24tB6X-n^0ZLiC%9*kq`2aJoVSrev5Wh0-g9Q==}R4|R+>Z!a^UN_<>tG1{9vtjDxNrBXRqD^#e(Lcg?qiCo$DG<*06If-Ol0tiHBAp21j-pwW7c8UBR(VDSAb_<MS%Oa0Ttb>{n{GF(k0mlYJC)#I>S<&S+9+-CYEo2DtJEZ(#<<a1vtC|iWXA{&SB@%FR}DXR!Z`@y!{#ya|1CXL#MZVvVnaT+Kwt6U#2yi7`3{UARUd}}>=Y3I{=W;Gk1_gIwa!SY)%IBRdYj7i0sdL~&Dg}G-SdRvnkjD%IP)kq>Z#YbUx<`6pKT8awl@i<JHyaRx4cw#P==m|X^xybK@|Zc|1i%}!g=UVJPhbNAjnNnzW|b}Kr&a~wdh2(a7UxpxY;(T1@&ebOM4xJhIj;fTmqbEZ7m+IF*Kxe`3aDk(cxgS$m0E|rG<x)r`yFH2wN%|%AyGiBX#lxY_&j=utPRQIC{S4w;3V~T)n*F^JKO97MT^zlvnHc{Q1;eS`y?_wGE#XYo$84e7y`};ihPIYdD>2Lw%8?u@rEH#L{B}(g`trR3SxfTU(<&=vae4=@`4N+82&_Y+GKp#Lk&fpV$>Ti#ZB1O~oBybTT)6V=lNuEhVI{s_C+tCW1QyA+ddetHv;Smnzg!RhY0KV3CU}L7ijWcORz)mJc|$GGBqfISqKmkPxP{ydZYMgC%Rq=kQQIhllp(q75x#0&Bp=x=@w2Wq8JQ(wi;46(Iw@@q+<~g%A`_0=uLsowU^ymCDdEF|d=m>9Um~YoZ1~TYx-4m?K|9!w#Z9%vToS8~^I{&o9!&ZLqN(t}C+;QWlsgNAWI}7$oU@k>NYqq3{(MWqo6Wyk77Lq!=M#AnGic>_@Qawe^CE^<~Sj@raUS0>fzlIT~AnrBVdqJ6Q_tAD$nfaK$sHS+Lnb#XA9-Fe>USaa+YV1I@(fTQbLfgQNk!wA4Ybm@^0#2qKsJ?;NX(6pYzfhSVkkz&w;VKdG=mx|k{6UvM7Kv-jg<ky)CnF5fcjX7CF6@pj*me$d#dd4SE(k_qu|)xbLT1F!{%1)fLej-e_u%M%*wIhWD`FdRC3p%lzOTU<moAEid1<-)cEP+Dc^T#P$_Tv_)9MSAs-Rc`f_eyyMh`M1bqaQ($~R2rvoM!HqUujvDt@W56X9t3EVLW?DAo}1mScco{QK*bEvG$(TFqUd$OrPKf<d*0P(bQAqbot*?MA3gF2bcPWcs?o&Xgc`j2veB&Nl&Zu|iXdozGn*4E@4JPqx|zcfAVt*99ZSap=rP!l(s017w{_FxMFDdHT0Q)J@qQG~fDiC)UY$3dYuWMu{AopysemznidM^xCgE;?svA@SA%bdFO9798#f5<|@vpE{T;q&ZWlKpQKMQCe^CRRvxX5jRdfJGY3k0ZF_K=q=B(u0VSnkqf0O&*eN{@DxD&XoiWpD>1d$fP4m2>qAd!qCUlbDOLt|pMNRrW*8VHY#mq?`luW%#_`^AF*<b1;(SrWyITZX>7@5}=3DHrh2$;ME9ro3X56qcNC-S>}-|hH2%c!$CAhgMctbw3-I6fyH?cfGRW@kvt5{0{xabcm{i561GHaFc(q*6oBcW__&y(ZrRcB7y`0JE?M;Gb7<Cz(%L@2Xs`p5@Tj-Z0T~JQCGbB0u3l(+=g~e7szVm&x41pEtIR@*0bS2vQUmlc0pQ-oB?KuSSxt~H7rOAJ77iG%)1zQ#F$A3NJ=ZmWj9{2({Lk|V!6?rd4Tqj+O!BO{uXa&fLK$^2VjECt;80UP;?7X|D!LepGo_EFs(}oz>4M)<Qc=tnS7#qA`<0tPm#a({V6fUjllVNz;vA$SWk5E0w%Hnsj-*_dfFb@ny47~>9mRjX8#$0fogY?(XW~-<%zV2#qypVcNX6S#AsOgd05V>lSi%{orfrW54WhG_76I*A)68KnHf+UI9jsPl>=AQu^>Y&z9XVDK<}F<eqXED_F+zSICFvXY2&9|tN@nF_7{-Z4G=@eplESz;Op{dtRBP5?f@@LTu&l+hz8JHQ7AaTpYDv3|-9<<eYvI+s0nrc5Qcu7ryq8JaGmv|LK<u8F1^5+G*-6{@zjJJ0LWt8NDqCqWoLK#$H|V5O+#cHu0M~@ur_nB&s*lJlV&D~7s*ACfxQucoa$V4*a<SDm34_biVl<zDm;qZLwQe*g+5^atqy`m(Aq9=9_$90_%5<aznMyhSqVu}6(>kRN#0`W)Fia_d>DYe*)XlZ4yC9wc!M<@7I&gJ>8#gJ+u!`&Bd^%}^97Dzg0I@zFjxN8|>vo=NpI<*uPnZ0YRyl&9+BVP1XDL)>04m}*8R~gbE~%#N%;E4LwlEj2X>%I61p5^u*+4wjMNsD_XOZ73ypmMfbZyHkkd@NXt2u}d2T27PhyfUKjvuqzgv54N-zfS;^^D{P?^)Q;FJ8RRItyqR4OB+9HbiPfs77;7dRJkIj=Vfk?k(g%(LM&HDIp)?)bIs+`aD{jbr1~q(G>_%>KuIGxwvT*heifWyqTh%P|w5Ys(3a^nx(mHTf>E_O&DNxg?c@;C@d>H+ilgj163SXC!PoL*u?H^U+iyp`xv<s>`s!Ma8k#l&52yhvTzhBIDWf~2?oguO$7;VAIvq#piV}4B`WeVp63WRl;`-x5U2}mKT)q_#UNs^aJ=}0yG2h$N3rV@8F7$-?la3LM=W7`;#9LMBX>jTSg~Kt=`=G2=oU_@9x92=o;<A82?uX20ACDV5=~@LtnX^*XoKt)WB1l*QaaNFY9tT}0u1p~M2A(G19C&GlR!aqzr`!z@iql&<2Zy(7WlzI2>U8{*9;3W_RJs$VAFP!SOsQE`fBb`WW2z6BAbPy^JVNbr%>DRklLt?=X=X$VijCH+g$7%sVNwM2sBI%W+ht$bs?KX(Y%QrEPX7`lXm0awp!}EL&;@QDFhkyg!KPx8>~cbi;QCiYb-DrGfDxp*0fR&ofXJxrRumvxW_c}4wFyKoPZ%dYN8=ugwdjd>)s|i*t8H_s_w8$Xh}uM7A?YHVjPMjOHe47A_}Kse?kBKfj0eQDufIiIBEjilQtMHX2?`>FzHXo)CwfyEIFcBBq%LTMFPrRYB7B^pSg7D!ZWS<)!{~-=Exd?)v7IviX|0)ewpjEe%%w;711MwPPvi)3P!`^<8TK4EQEIA6&6P#I50K(%R5sd9EGvazl)cMHfniDrK-VCT4x<s6`$+7&5NrHR!b&)83$a}`#%{WdLn&mM5`I4vNnkb+*U?qjh|c~z}UrhAzB0JECv)i-?;{Er|VF1#(Le0F&R#!wt&S53Kh1=bL#Gm;?}Y$nI;?h2hpTky8>buLy7X@=5uZDQuo3V>|pL<j8tg>WVgSCy5QQ?5l|;ib9V`nIJb)>qf$Rb^%fnw9)3hNLqok@-rA|%S=uG%*#qff2cKb*r}L(k3Fz~Z&Km=079(hTx54HmkFNrwftfg<pRLpTjw*;7mkj}C8-<s?W50%=U^UvBt%@}@7hN>$Ff?#WOk9$Js~Z}fT>@6y)ypLWP)xV86C{wPV95r`wAFqvn6UXk*?H4=H=dcnBKQ?GD|BwqM+~A~=SrRYF!vjlKty_Itw^L<Vo#TTy4FBbj0rU{u-8*S90&RP$K08dtz(UK!3>OGC%y{jZnd_zHl8UwFW?G*Fm{UWJ@&aaXz%=>gM##M;SDug_NuGCpvzYt-Bv$eu|_C~B7iT3P<HxDqyY^VLt@}J$rpv`W}bj40aiIeB7i=#>ZB})z%gei#G3X5h6F_klBJ+Z+azaDkkv-pZ99J2+o&p<XWOsTJS*b~4WQ@WVpA6NvnnOZID<V1+Uofx76i*gNpsXFwN*W$u<PwLT;1C)El9tx-8br*uqW0U>^zw_{bCGOc$Q&(knt1pLx4J-dWr-GK`;|@ANZTw==sGE2i&o3`vk}!5s<STSIfuOl;si;_;dw^BT(lBQq&2W&_2OF79x?>3Y^B^34S7+=6*t;#+-t$=2TG8&`C`bYE?AZ^p%uF&AT_0ooJUrD^JzIPTdFV#2%&+)7YU$ztP>i=$)s3+NbVOr~apu5{K#Au$X8ex3)nUfNAzTMK)6+S-rf03!uo^h+pWbNi!YwJxgl3#$VQOcSo_wZ>>00TDBE%z2m(tZ=d}k;0-~fRSk*z13A^6il>F|kGdpH6?d?fOQ%&Fx34(%TByW8BvE*3ovb)4#3~7==$R20ttDdrf$~QR^IwNoV60tNn<2B0V%x*G9RFDDPA5TlEW~*dq{l#xC!O7|__gg>=c|;n6k{4NzJMZ2bX*BXYfKSKd)?M4_+v&x(Dt8FTna|yfdlBUPGl5OQM<`e|Au#4t061zJ=hl10eRq3x7%+EOaK;;^SejxRxALXax`-WL2J1d=GNxx#J|!sfYdg}_=618<r%~v^oeUPfYCV=Vq$O@E54L#s^ON%Qpd^kG|#dqRXS7(vuT_aj|GhJih^}ywR@G-UqxNJ&;<$QLU$AYT);p79{)VcQB@X)xuJ-2iK0+!k8(t~VB!L6yMWVQqW!xf)#^o!`pEYhecqPsP?=XklWhGQ)sOfa4V?lJ%%a4c$z;|<ml-Doaos3dtT+Nme2$J6`n8MYUqP3buZ{tNtPP>B`aA=jUA*!H#8eSh)s$LBD@q<PWGbW-?d+k}&tfd8OzszMzjDJv7dW7Rm#AnB(rH>XrwULchfb8vjW@N>wX7bWRc{t&CUGCnYWbd?0IWbm#|H;~mFyhdi{Lcd=(e{0bVJy7kVv&3K)@Gk?Jc$IJK_MzV+M8NpN(_?W-Rtek)^+I`}S*|hSoX@eHwzlH88N)gEfgz{8EY5p6Pb~Uk<lG=GDi`CKhvWtXWodn;QMbsoI=b7phUtR^9EDRtz6i_?sQ@0w3DOl(i&;yiR>~N6iqBrk10{w5odtma5kodh2u)YL-H72(|{TMG^>7c>M+gJ$)m_&9GSuv)tX*iV`uhZZk?*dF5R?+V<K;|5OfE!uhJW<>n^KR+HTJ#oD94ylM@vVLep}sM3?un6P|fsxewLPJy7M;X*=nIr@w48s?l{&uPWkT7l9^4s|ULEpw_%A$z%Z!oglL!h`Xf-ud%hQ}t~PEJ<Kal7pR)>;MyppjWxGd8t+BbQWNLr0e;SJj^C`<3fUc6xJt_#l0pL;&nA#mccT^OPN*;T>hEtQ8rbne&Apnx&pqg=moV-?6RVuv8kt@KrONrlU}V-BBonxzFu1xgkYx%#&Ir=({kjKwr@(;!~$qUn2<!m#fk_MGwi80;z<nExV;|FC8p*C&cefwS(Jl1Z)~(GN@9r-sc<IS(Ia-iUd42-EboCJxML=z@kV9hhK0G^j`?&u5U)5Q0f}ssd7FbcXQP&jw{VL!ehY@V&4Btv7){|hkbyVcGGk?hy`BrOYbWO79HC_qCt1vEPNS3<o0@<czBbpfO>Py5qCvIer5XYH$8f6pAWmbp9eyA)@YKyoR0o*`p}fo3fStc^7J=A-VkqgU8FtHf#_0{im?Rr69ujHRS+T&)B>7(o6lygd4}uxKAW9@qKq%N2Q$@QV;Syui?{OMyBTAWz8SSaZMj6ph3H}nXMRRg8ekg%FVkxv7IV)P*Ae;qc4s04J!dw|a>h-EPqi`6~tt?i{PRO2a3hM^+(rk=qt!tb>CmNMH%DRqktxD!_s0y+Id{9+Nl8)*N(yRDLskE#c88MB@l`To?=oQji5l@~G<%p-%%s>;%z*M{)bK-~6VauTKxI9~00H=#W4MoL*7s#!4N2g#Wx>T{6^)io<hQJE4wviXlSe-c7d?otW*&T)I7XFGGwuO=`?$u@xEN|HSM(x@)hAdo-ZN*diE+VTp-&QO0W25XUtM#ESah{&td$!C`rn<VShpg7&^+?Vxadjz=7o%mXvBc7CDv|x_@w9R@$O?M)QtQ96uRFA=3uUd!z{s*PRj)<Wvq`{Dh=Jb=6hF50IS5{)#GZ$PMxEzRq;^uFqyu#UQ8gy$viXgzR2cKr#Xp-A{e*umvi7WoePxwVKk3~_Kt^V7mkyT!VH9Ejm7;_}^&zBo1<x3r_hhH_V%AwjOnY?w(;WX9L@@|5eCBLXuNVSF_IFT`v~oxRcRDakYlW2Xmi1841Y3W?rLQWL#3`0}Y#pwlBGQ%it;goo{o_5k4wo$KR=eby4YPa=(g?>Rw)+;o+SW-$*Lm#zC4KDs(cDTj&qZ=2WD?f?kSFky+dx+{Gtw;?hGQo#q!6yOYNiDeeyngdSFEW-br1>oIt^?xl{B6c*s^CT#U<o!ei@Kj9kL-$6VB`o9mUu|`8iDEh;yHzN&sQRU{i<(lV}YRi%mmd9ph+`QnFym1TB~Zr^yO|ilEWyIG>I&4dZ=6ftb!TZP4FM`g?)?UZlU5=<hT5Hy0oNKza8GA|_*r{F6fb+XqWz4))ujDe^t(O%A7+O>v6j&ft{VKAxWi!D&KfIzZ6N-m92gFv58s#@I$^R6yQ|Lb{BblW8{U)BR!l9e~O?|6FJ9igO*|u55p*Qmc|^4CA_>7*SU?fTW!{%|(7RAp(m>DJaGGG)s)n`$TnU%JY|Zp#U$d1msTL0%O`v>X+-{!YWEAa+&Ihz(K(DEUTppwHkJOeT>PKX9UEq&&x^$Dy)LucJQ>k<1C8RZc)-bw>hyT%m}h{%-&tkZ`ID;>C@FHMx;p5Oz>(Xn3bq&2BMSQf?abAmds$j*ilCqDcZLQM<Swvs&ApM6l2yDuMzC^Gfti$jIRx;RZ&tv2pLO15Ppg|iZJI6Qobf?0f1zTO?3|E6za0Ret!G2@vwJJaejyr$<OtRykmIaLDi*%B^#CW&}flp*fCM62AqvkEp>ELIrwCwH5x~`B^4ueN<-pO_dS_pwGBXPc&vjsrKssVHI%3qHsc|%4M^L0bNBu`N8X3u{qSqE-8<R&E%<-)&hU7ny?L^3{Cn}F)%?s8cZMx1m#=h{0H4}XQ#B~({|3e}r!Z~;wT&S&{!_#Cr>5&q{W&C|CA6?}f}awVZnkcto2*BGyH9lwxfM-8jP&T3dpe~v41`3Oj}CT-gBV4tOcG!k0h5sUnQS!pOl(ylo22WXAL%6guuI`@6BX$Y6~DG=X+-B^hjapoa?nWyTH9fi4S*cO874WZ9%5dRkymzTicpGJ?a!NBm*f)qz-M+bNi=KTEz_pK^aj|7q_f7kKY{=<m_%!s@N5^Ra80O-0yLzlD(4lptw^1Aa-WvKtXDF8(-jZUI)Tun#sslGzPODogZP7Ka%a$;<~i?NyI@mm*&LHwf0kf_0T8~@Ujhry{gOZpZl1l`*1|_@jQ9aIw*6T+yFMznS<En|RcKNdM7Nq9H8l^QE}P`9a%6*iPWf4Aa&KS1>D{<~|64b`KYQcqP5%tRi-?sxEr^yh?Y8BNyJ;^E&C@W2d}zpYp$|%)*#peah{IE=n5HS6KvQT0B&vb}On$A7xz9*(mE=DeP^uk1dZ~z=wiMI+sP>hD&ZXq~pZ&Y5o~#|&DGVBi9>l3BV6I5yqIlY;gi$%b_q1bgY<9zJ55pXB=RP<%=ooJddL<(T-7r$KbwLKg3{<6aeZOLv)phM<SqAdaaCiq2;O}A8648XS?0~n#$JWj)!>WO!G1-~)=*qj3YlkwY139u%G&B%A-d-9GmB2A=>*6G}Tbx@)-mk4IsGc&;>l#u3MKgHl*RAO3qawDl1ZiNwzOZ2-jX*i(Sg<Wqnxi!R%xJb%7vRXf*<}dOsH*2a^@8)>@=S5Ja8}iN@f+e#8xlX`J0MB&HZXiaG7Kh_;H?meKG-KEO!9K6?ggOtrYd!odg<z-)RVEOb2B9P8N*vrlR{EeYnZT_Muh@X?fSOr2Z&wgbYKYq-1eUo3J1?ZM-v*3xMPz8BuA{WnZaxi%*ZC2G2R$#1_lw=qYH=}x1oRM9OT9oSEV(F@WTKsI(sA$=<H%pt+C>f#UYg@SxGUNaWqX#`Nfl=F+99(fi~th_vSTmrg%cx5|HLB8ri(3!)JZa(9vxApZvFfefaf%`S@Eu`NLPg_kVu-UvcH-)ervalb^kVuUaivI9YkaUXh!OH|V>C9-^e^Zpw^Q_9&OeUB#O+Z^D1te=_#MB%;-Z3XicUUQ_#-Xj&-B6$!6eiKlR_7Zq1dodR*PbU^7|!%1$v$KN%FpQ`el-m47i0-T)g`YmB4Rh7!lp=la#O<4=(o&vrNtlaURdhNx}tqJjzau_j%7{IEr+xiNuDHxnbcGQf#<eQu%y#jX-lU9;t*~eDMvjYz{zRhu(aZ4^XJbXpBB@t3##&32KFh|F)l!Sda>bo%aSmPyjN+;l@1>!HlLE7H(y_R~`sxf=pC@WKdAhDLUk1-guXM@%MD3}XcsjX_x*<pETG%hNyHxr*j@FhOewMuMtRk8X~5Eo#3YtxF?mvSm%-#e|sR%2-zAy9278GB$IF0s*4-I?)Piry4bDc~5wBt%~lk#Gd}_UJ3)z;p^KESqXEqb#N~0DCL^Gx0wE!sdmIO<sM#qiaWL3|GDlSThk&awig9g$9-uc?$-X9{hj%&bd2&|1-_)I|pn1r(1rRexT-cttn8*6j$Feef^Kn3Qz3DGZ&t{*w<$;=*B9t!R`X`MSja{UtO044GHkhF4#Kecn5{{n2dCHh^=aVjsAC4pJb5T9h^99Yj<Z4|2bU42V-@O_Mpthv)v0_-VG5!U=y7$)EN>nm*W^9EGCt7G(}js(fC$<2BHdBH7%+x8L3kU+A(%C<w1Ufuhe>xYDhAH8%eUan7dL5RQjreezl{)xT&wB_-oNa;;TIh2ubMnselb)JzrAN(#>y*%Wztm(>DN0UOq%V!H$CB7B+rQS170?!Eg#lC9pT+EIF&ParK&5d)Xd9gCoq4RDs-ka-=B!Sg<{SWiiN>i#Ha9@JpF7+6>W{RAP!~anAHXdkqQAjJ%W5eNE;kc~fphagRXZ3e+*s#$IoPXT8u|RhtG@FbA}~mF0LcOOTC4UDgSRWC+#+NzGnkG{Mqz6%b$LWuRfJ6p|Hd!;4b{mv7dZx(<g^sck9I=iJuNnto$viKlPM<1CfU)M(|PLIq4l$@g+XWaVUn5!Te6CP!;c$brPx<IZlf-`St}tIjb>$Y}bhpUFnKdlqtHr<VIks!H27*yV8hX7e1$=q${kvy$jh`?kTu3KT+)4wAaS_`FQFF)R?M%AB<^V3}a$KVw6-V%e)GmW2}OH1InVO8igmN2*sy^dNB;GNdrW``j`H&ht?Ujc28Y?@1Y{9LfL@YT-*$Af^v;Vu%YpEzMFEmeSF38}+fBUg_G$wx{`DPvI`BE}Q<c4WXjb^l<jE0{~K~BG^CUEBAdFRF~!hchmujbjrxVKJ|_ZLHjFooYkbBR+1E&*(#EH%L<aQj4tKpaj_K92e^u|R=ZODZ&@73x>15GE7oiCVgfP`=#WTPi4NU1OM48?^t#*`QK@5=)~Q>ihAD?jak<YRz~LA%f<*)eCdH2q{F0Nvu3SXzq2q^A-C_=W#}yr}I(Qh0+K`zwuUIJVKy8bI1u1`^&bhX|lDt9TMGbWoLsmq8Aj{AiK1=#<QkqXtp8PwD3m3ax9}i~^@DIFrZZsP4)vp6}99g-@>8SHmLTAvGKsp*ifjUFVwSY0j5Hcjk96)u=*?Lab33f0%dOi;29<Ty_wIZJR#W7^oZK|1lo2r})*0)ZnvXpseQ$iV`+i>d-0^5N2HYyk4m3T8$OpAu`gL;YZr~|~>x}tx*PT?Ki81veCcwhTv(8c0RrLCiwmUt&6U)KZ#n+2*_v{hi#RoYP$g41t|7b5IBn+3+`ny?sO83))JD-v|puTYx&?GL|p|HmJG_x?}s|LUW6Kl<f|e{ugWKK#!8Z{7d?NAG;}eqVXK-GKLF3$V!{%wiyX0)v3ld)7Hn$UPZP;VK?trZ}yeU79gn(daqP-~~DEi-tvaF}4Q^LKjjni!l{jd82%|Q&)AH&?&Z&P~y~@mJwam+i(^&Y-p<B#jX|kgiblU2Lmr=9iy0qemX%Hh~0%;$r!&$0in1X>%xkoYfzObFD7&brVs10A>|<>(0q-05U}21s_4W%>MBk7lG#Ay--c#`1uFpb?RGcn0Eq68Ei$~gt$<LdFSfQp`+=~gu4YYR;Z5C(nm3J|%maH<+ZUzdAVzbnmd)dV#6c*aSO{1MJV*fnc6-Rbk{t@^r2r|kzhu*7FOqAZ+I+_uCK8q)J&5>@=&YRSb!E~KcyP?w|M=J6eE73(Km5(_9)9;*4}bCA?|=Ia^6-aW|Ml<h{ovzo{q?8sfAf>?{N*R#|0jC;?R%em|L-3D%|Cqd!S5b^{q0YG^_z$P<sUwM|Ht_4;~)O#kAL`gpT7P6!|#2S?BxIJyPy2*M-Sis?Wf=PX;p0UfB&0bK79KJ4}bIz4}bAP0{!3Le)x@FfAY^i{Pb79dia-LfB1{v;M)(sgP}FvZ2aJZpU)xC=E1$Mc{}(2aR!OL7x0RL0I!Yj{~TUq_uqxZ*m>~5dx<xNZ^k0yI`WX!V$s?auVg-mb}rN{+fD)|(3BRWibraPx#D#$UaEtD32M0@9-fI43z2Dauz9duo6CS|LwkY)g-xR*a)LTQVBHL68220`gVaVniz3`|SuNPQ^3{Hx##f2WAx~J!6>B>JfP@63!YO?q0nL*0$wg6;{97wAU%1TWQg=D)GgT~Gsw)E*zeP#iw_#ElVy6Kf$N-Viy{x90^Obl23Rg19uj4g-a2NuvA)kQhGQB&^k~NAcrKg-9uRL*Q8`_FTcdyWk=kHztmHm0y+Qr&gH|B&wYLV3;kuByx0HHz(K9WDjDEgExe(=B4{LB`lM7ax@NbaH-3s)U+j*z==;!PxQLwthF{w`LTk!e{5J-qvecmKy9{`SYe|JS!4{``l)q2%|+fAiCizxlr(e&d&Sb36s)?TzpM3OVb8d*1_2@Zf`Y<GbqB{ePSSQ+@EkPmVk)+w!Y(XZ2VFVkPa#M6jO`5CQG)d<4#zdWf>bn_G{P1qBq37jP}sUCI!Mj2d3U707tEAgo0+qrl`h!~KwV2?7R(Ircy)zx#)ue*4q+zxv5LU%fjB=WyIDw3nlHJ$MB0Kd3T+ZGTs7r}|c^LVwUED3atQBZp`TCC4M#ARMrfxTyM8!#>T#TF0AwSy#jI`IHotEq(r}q$_PPWVGo40cuChtYl_Xk{*kATUtOYvL+!)2}j;CWeN;VH`(=;V)LaIEY55KLY^+>*W&6)IY3Lhu^gY3JWefe6|n}!=Q6<$0EvDej~k~pP04O8E2}SD#M^VRj4#Lh?fj?ZtfqQSeMAS8bl8r%G&7@R^_R0}(xnccwJdIGtX>|6nZ)Q`!Jc(#q4k}VWt}!p^5v<D;L7Ub$zu%&j~YWxJR_XQOoFHL-x1VRli-!E+|#!x_Ef7ziZu0LWW(dH)PbJZ+>WXTe69N>%LncpslTeeI;vtGTXVuPT7GO7HZHlSag_@ipY(32152!Ez}?j#d|`ae`WToxIRjBD@rXr=t(Fc6su(U*StcMCcN(kR_Nazq8aiYm6wj#~x~-g8w8tjLvWiqTa+X`?EVN>)>+>Qxp@Jh;-76hfC&hbk*0!33=VZ$}h8HKkWNB+Ps*v&CE-{Dqge9z;Hbmz#th?cO`Ngs}Bha@^;VYBL(cQPg)A(kI;znG3RfM`b(!O8Tb<eVvcTTzB7j1#6=630J;QrFP*ZKBx;zO|Qz#cbLH=)Vl`(&;{Lh%l@t<tr?bhnmVwACu^i<39FU0OG4m#^xx(@Iw_-{ZR(YcA8XuV<{eCyy^?bYFdk08Z4gKi)WGA3)bJKQ$=>F0G%siV>gny@DR@SLQ^?k_}Oy%jrTipKh*5FyHqVU%K`B^;d7=&c&Lt;|Av-pyz;Y(Z(j#<<Q)IH7v-O&r;ufyfG^Xi;Ke=?mGc>5i3K(`SWU5v@ZN}0GS_weliP+ab`zZM}8?k4j<$|eUr{_aT)c~1rxL_2}n7fZ;3|@EP+Mbj5L68Dmp}^S%?VO+EEB-e!z$AY`3q3#k5X%-DSrX;$nG4?l&YtZ0PI4V4qZ%Ns|OyP%wTYN>q!#jPQs@!|4FI$x>+VvWHg}hp=-T;NfMDw*owE0-KD4pW|5*IF~ZRFh)hI1~+3VL3Kfq@&TEB4VOHnapvfKx+Daq&dFK$>SlIxiJ_;b<U7k;%w}N`!wNgOalbeA!VFmfErG5xE9QO0yS*vGMAfdj+B*yPBl`mHqDu8>E*7r2##_EU;w|3SicxBou;emtYRq|;H_BVGLee86rR|dI&6jowvpb6|((Fvtxh2c0uS!l11hF{3+AaDAZaD#z7=zaW0tTdp-oS`6fk6>WH3vo#EsZYnw7q2(WE4_Ia(mu2G6|}4q!i+Jo9=ad4UWLPbQld5Kr2nGjQJ4ov^PPM9~|WeB@JbAL5ax`UGYOFTkK5nRHo8bFz-oOVKGr!dTnP|f%A0HVOwBR>cZJ#8c}|k5Sb{OqlZmt9Qb@WaR);034jhq%#?49<T=?UY*Z3qAFpECsNBPf-SM9CHZP%hs_Us9;x7vGla;FCj^jdV-TMHYG~oZ;S0~K3)VO_2pfX%c^sUvZ0j;m8C_@}U`I2h!Edj4b34^R}otq#a^1%;GQLkkwJUseY68aBw5Po2@NJ$U_#z|SxnuU4^hjU(iop0iU(|P-ldp<)o93&rlhZ`>EHj6QZ9^F&|;eCKj(~yGy;L=7)y6(xX<RHMO;2xln>ytmG)<=i}9F0>FQRX(kKw8>GC@9qrpWfl{Q`i88rpExCzVr%?6~pOFvMrYMDfv#!AL3}smjcOWnq32Z<aHKBd(Ez;)0sOmnKI$9)rHNH-&I0errG|WbiJE1;R~0e37}{f=@X{dNKk&4?>Al!ZoY8i+RI1_t~^4Lf~7VC3$wXfz#c$66h?Pdl(GSDwI1aRA>Sfk$YoF^iK+6XU!lhe1jVOKabK-wO{ua)nTye2malT-1r~$lK-A=UXv@1BuU>mOc<DdFDQx5S%@_X7&O~K+M{zV6F24n+2JOy*dS}C@H*Pxd$(dm8l%;v&raGvOB`~hItm_e3YsRBllp*dEH3U?_<sQ4j2wVzh#8q5u@lwL9#ByIrOg!Z6?(ne*i2uKI*@MiMN%f_<RLM^3>6Aw4TH88tbY3HYrPdP6D`=k<LT3RVcj(lRIQh%UBIhkOrCxt=9d7bjCTyl<L37J_4=5Ws$?RC%vDj30Nm~vjTwuE;oFgJ}nG00cZ~2Ao=dLAbx|oxQ3JRdi)5*EYAUJS`)C=}#ad%S{(NnQWOQ%dD_Da)=r*as#>ZTNtOiRf8N%<ZS_fi%dl_q4YK*>YUvcsPG4I^%@9+YVqv@C;MpThJrI$uYo8A?2fXXuLL0t8Rjvm}w!46n9live$!r&LFN$<oPuOyZCa&}|=={0L(-cO(P2Glrx?I=|6`9}<*DIxPh%af4>ICW<^9qQBeQ=$|?q=#Y+D4^)eku#MXk_&DJ5pxVGeIFQMj5cVYrb0IFKI2(XpP#C>1w|3E?R<$(H?R)_!BrC7$s27eZfI6UqL0CkjOP>YNp0E(j0bkElF#=gbF1&LTU<{(oTiJCE=jD_6V#0=Rx}Kb-S9TsS@k2fKa$%gHET<V!nAM{?x4@{*&xHzxejnY4Y#&xz+vOTEh$0u3fZx(1@r$A0=`H<(@;6P`>xZs7uTYRg`AM@S;jjjk^0BLWDmJEJDa0aLk^!#5VOE80Xm7+mlsazq+61MNpdCc<?wDB@_UUljoX$R^bc8nr76RJAZ#}L3LI6?71~_1qKut@pIAV$LJoMS-<E$WU=&0da?K{{+t7LIFHrJP;90{RXu9b#{@xAgyTID51lFL-0hsfKtd3Y#`ru=HWcBU&#f-=CfW|M^jriohEbTP!Xg(Y;LT``y{7;%uFj!jbV)P89%`6?i#=&3;Ut-wU#E1@R>azHBk<Sj^}5;MYIBl=~Jf-CcG;b_Bh;a9uMuxvj++VUk@wB<ruCj!8ypxEkoUsEeCOuSl6;Q}Q@4ocXs5d|pDq=pI<Me>0K4xngWAp170gEB=aVxcTHVDwnd2U^AX!fFR--=;_=I083}!<htL&}mLEFw(oGa0n$u5Vu$j==`}$JSSLZn-bg)qIlBmJa-8g8ihRzY^42-{)#Pt;sZDoXsT1(<Z@w+5^(^{|7fm4XF!}$c0UU9;<<sU@^n(;?o_zSYG_B!3<~E{k(;+706$y|Vhr8kPC?9V1V>LjyiOcYj8ra2Bt%CMygAsKJaMvRmn97w=3=GxIX+0T1gXS51XJr=*7!x_Xjm7K&<EQCQ-emPuOWd;C+%F=DVd!s25ifc%iI5{!b>@e766M?g0)8wV3dnPgK?-Q*mXSQH%?;AldOxn(tP6BwQK{~Pnp5&iMy7)W-;XnL>&{IrAnhKB5aIkCB>po?7313HMdu?lfgIT!v`Uk&VgWrM34QyY9|FVK^AoJrteMl1x&FsvLDzVYGo9pYp~Y_U~%7xkW9sAHWYx2KT#?_0i__q#Wxf=mVUWdTLx1)JmEj-z5L*V-(k`n2+w89j^y0^V*Em!ymI{(of`oY(z!M7+W5h}w`ZWk8kl*4m+W2!lDYp5h3)+$m+yqeCzrj!{eQ-pd~ok?;e)dEYsmsE5a&u$fGyGGTOdW4RkUQ$2dH^^|K|`?e7G=LEVd9V@ZjD*QCRYiX6TZHVsj2wR{huf-`#_24?g(&nRf_?Fu_<vvm?UMup2XgAA7?Ew6XKx-hToENwu(S{52!J{ovmJgeLC&mEB2TAZYS{(;M&6<k7Tipfwv?f2*_kl)IUm<NN<I#aa7}Wb#w9Ae$$QW&gnkKLeFCesJ%7UPy0%IC1?&Z}-8y?+z+peS-SltgsAB?!Qwcn}wAhfAq@`Y_o8c07h^A&f5dnU&CJD+Xwgl9&iKVJ3HRX9OpTQLhr>eW-dnV5Kv`&$8cT)&>y`^b?(1AFJ`D@Y-0=~WROXQ`17AFRIDkW2dox|Ebs+Sc>>A|AkGfDiXankR3lQ0DtoS7&8p(x_5MS&Lz#%~;!6HC0L5jy>%Btl-1WXt&;=GTu@-(`RWo-NsqC(|wzlTc->v2U_=SRMN)g5x9VBqYC`ae+ddG(aqdxKYx2d_PcLgq~H-&DXUl|4&^{UW$;LSOM%iw8$?8G)5w0rVNm;;z}N;`M|-B_55|K%@#S$1v1MW@l-7K!_9a?j23-e!xd`fX>z8;{z%2Aqd!3vT`V=$8-fy*Jxkt=ZD~KX<dr8dQyQ^dbM?jqA7nJh=7JYp>qA-I>z9q>tS4QL>6nuSUed(DV@s!M2SYE|Tye-i17}rT9{@j;r#n@Eo_<y(-C(Pf>zb%C+r?m8y}b))M5h_qc2{PU#L}U6gyE?ANX2M3?ona-x&9NXeHcV9J6-D-cUX2}9U>YrAr_n}dC(rl$j3rq|MlE>)pe7YXHIPZNi3(2dV2rVXY1803O4t_<O8nJ~w{oCdo7qV~h`n|8&WB8v-N8tACQ%K~GBXNM)<Wn)*4V0B7C1|qpEbli@z5j#CN()CrM{)(cH%UQwzvwEKi&c$>OMgk#4DWw7_JoJ<$3cT`?z}xInyT-?4M9hyDc|7T0k~jQbj13{SMH9ehd+L+`$-Tj4q%c*kjZqTGWFN0?Y8NPHMT+XTffak#b41?NF_bbNQNQ(q8S5lhy0~I<j0vE0YL!ZD&%UTi8PDksB(zG2QGKiM6KN%*X?X84<9u%|3a#wGQz4R+>59*ei&|ukK%8c+&h8}HY5M17`71}oZO<5n&Y2De87o`iCGeTt2;{yokEXjq%pLA9n9vM(hah~%b^d(0I}Br`dY&Gr>O<lEYLss%#|uVEUNgLN-tv((+}<{dw>O8n+=u1}@)Gx%^cUMLd8_p85#q>OJ{~UJVcK)B1$1o0cx}&~!6oVlo>)6WtW1D)reK)?bZhkshaTIZfolE%=K+HHV(!BPuCxYAqWm3B+JE!uA?k*C_H+@B(LYN1h|@acN+(fsb<~pM7WJ5L>d}j~XLqGnXvNhtA@+_(y{28S;kXE~8r?wQXuSE_ha8rL;i(8bl<`nSQX3o}n5D}Oq~6tZm*iWy>mBp7;{FTrUN0aAY@oa@JLtNjQk`%JXF3);{42vskk|~9l7)1ow>l?jaGN=~mv*QoSA(;`ForNyX9DN+EtKvw{k1h_kG?*z(8Ii6;W{N#4@3wD01#20*TTTV2<{G5n}ei0c(4%@PsVs4Pc@4tUTP~vbhl8olCoA-tHgLm0cbx~wQ)3=_x$Ua$1Q^ZNDwa5EHA`oe<`qJ?tldcf&`X^;!#$4zGJ5rVj6)LaSa4GUqQNsZ{(;H-qNw4!?L5rgtlEib-E^O&;!sI%O+QVpeCqWf*d4)Oiu>!wX;D_FrXeq$b9!W7FVK--RoPsRD8li>OcTmXx8e~B85Kl;cD$d?ZlHfxx6wPMtNCY<aZd_3*m!sotV+^DQ(DEo;7t`r)Gqg>W3(RP+U3%3ZCOR6$)OHCZ~ejP@X#-ig-5G@c7W#y8?0PYoH?<vtEz3W}IO??lOmd6y&Y8$gq~S#aqlN%v9)lC6YO72e|{H4W*QhNwy362VrMTt&G_x)%=1!Hz=_I%0Q(eiWsRt*HOTr90p<p4su5#mcOO3b$vVM6c7CGfU67+*yg}8FUx(gGo+K<5MTMlz&bc$9W4rDQKuwHqzM8_O%aGlgR!$&K16*Tw{JX81V~y{<{B#7U9;8rzwr4_?f',
 'diagnostics_snapshot.py': 'c-qZe-H#l{aepVk|Dk<CFgw)V9*?%1VDAH<NeZT|4~ja)h%blH>}>Du=w@fu^Ko~p%K{1XVB{e{h{OmCLx^ZWP-Mr56(tD5d&t8n{y#2%Nq$w`J<~HgyLXZWClth;>5uB_s_Lrhsv2%=Y&=M%pG)D3Z0bi*S0tlR6vk2vLw_76Ssn)B$B&<gQJPFKC`kR&NTfyF6=57i#W0M=V&<oL=yP+{+t}E+eJh;Kk~9~7I!0e6Z{OnSK@vqW$ipPggdXuADdJqFT``m+zlic-806*jAUn}Bhd?n8r_xL_?_EKEzn{dioHWU&QTY-k)g%8z;h;B_xeow-+4WwQ#CoBbpHDExtn&n~<*-@g=c6Q@mJQiFE1%yFXQMCz8uTAFBS6-nN02)Nvnx*hGzPIUw}1PVz@I#w>nHk?%DhNp#-tY|{xIWNJqD7?v%DkYAR%J!IYmC&{1eA@)ns`V$XPBrZ@&E~O%o9EPyMKn{Ok%pqkd}{gGA4t&15|6jGS{`C-$du(HG|!y>Q$mw9XeT`-)a<EF6hA$(fQpc@}1Q)^V42dU&Lo(&RJ~d*$nCm`~Ky1PCtgnzoeXz+q~E1XDy@Oa~-KEW2SWBpPH2G3lt`U9oLrp%JCgJq!qEoo4SckL*8$wjrX$G=k^4%TqL$D&8!!#{RROP3LN_O@3*@5@r>*aD=?UJeOIA+Ph*D%4k@!N|6TS50l>8ph7%;{AMSXr?ibd#{fP}LmW|z0<hjg9MB(eLrGYMNvLIe3Vat^%Tqt|`K<3bArwg}>)?9)Gl4JxlI|fs0!?BcVQb5+H3wat^J6*htJ5+prn9VbLX^e2aaN?#^Rplf_g?j*OuEG8Fbm@>_v1jCf#6h@L+fBSt}7%43hrs=M=51kwyOqeBzr(;3IV58&`GU|28%jZ6s>w=ne*(9+hfs|9VOVCSvV&87`8#Hff-&_Os9T2?<56qjg!;m!zNE;oYMiajwMEmE#XATICJPBj8BrlXSLho5sn4N2m(|%pF^U*#Rm-^yQ9ql{f@&;IEj0E!r^(2@Z(|CkAs`?YI+Qt4Ckn(9MpQtTH)Lo_*s}aec=?bf8vLcKZqpobudd(^A9JN{|1v2Tz(<2L6J|cE<Oy!IP_x?TwQ$X6L>jDl4CKwy7)!7BaYRCteA_`7x~r2vp{_L?CRp@^!#g)p+TG`=`qC1Sy<EnF%*`Eo7)EYl8~nJ&?2rE4$UrSX);b_)<)1Zd@{3EF{z=uN`lu>V4_z|8Y5_tufbl-YB2XEYXKT9XK^=^L81_#4kGBGBO`y7$)TtF4C&}7w$AK8{>5f#`Fmup!z>|#?B^ZL4L^5|T(_4exgQzGM!Udz%Z%7+=uCJJvL&}z4V-Hs2$Jb6lDQmWhMKA<P_}#dB+R^O_h8svbDV;S92U{i+_JsY+2nfdS!->TI|gD^Gx5N5?2Ee!(>K`du}@D$x9S|J^*ug?7Pj6gygiR7NvEsM*1pBlt_72cjE#fJePK}?07hXPX4o7$aQLDQ@!3UtlgoXGQUQym-@D@5uDwD<TtD|BA9x$)U}o6~xD_S)6}4?hMH*!uG?52ITCnO+3p2&<i0|H|rCIQ)p`>SRy9!5!0&pdWZE%TZT@L|{=F-lm{#gf$S0b`mZOknpZ&#3+_vPp?V+BepwXbOu<$<l~CNin1N_zf0&>A!P-EXfhKF+moKfZeLX)NN&)r(K^9i<*8$>m3wLoH&6iBO~I<*#C;fWxa7{}rosdg9T+eGvfmUyJDSBSCO~5oq}FxaVj*EpKvAgb|pX29xkaDm$lNjD@q?a8$@6r*RN2el(-iVA2tLm5#t`$6OG4MIIOrMC(<{@0|K^I8y4Na@@6oHhrv@TWqIK+-o(#s%w=#muEBFhFD3QL&3t9z<~5cLq#hX3v<weB&B1M%AXWKNo}E9ZiD8Lr^pi%I;<Uoi=OJZy>XNbI?f%n9k=eA)0i3!8a{qCR$Fp4rQydy;&U83SsSfJYBNH%gQk-WGGw4VTVv?Rx#C-U<$y!gtLOv^LIaK(D_?WLdHiteou{vfcnpaS;UT>Eby0c<`Q>loi8#6ZD5(pz<6gHpI=}FBiI41;TD!ozYdlBVY^M&3w$f!~ML)x2>W3s@q)uVLbaNi&$_IkV&5MklCV4*ds@I<seiY{OvZb+z=^jP?n8ybbKlK8-dea*_>p`BR{6e=Ar$uvjYx=*rgwg-p2VwJ1J)*mL&L;l7SH8=e!D{7e1Bi^rxOmVgKM0`ntsNS9Z)>U%qXDuU#QXA&!+2PmTW)tiHx4NJ3chJ+(AqTXnbmskVo4$n4NH&sOjWY2n_ZImcJZ!{Yk{n{pxH<ZM7S{9i<nJ1q#xXN7{d@V)&o&P)<FS-)kU8nXoX{TQ;Nco+k?MEK@xRcs;y!MxlPx`(t74h>xJEV0D%2g<zcTPK|@!VP6PMcVc*AQLs#6nqs}ygZ$HxJ_$tx&bKD4LdCMtP65}qatD)K!DQbec7x~k{&?m*uRzRWPHr!D-Xj#o-)NdCIQcEK3isEz2e!sR^%J0`;v^uUWWXuW_$CwS(@^yw6tlc2=j#+mb<pM`ds99=ELu0A2bn0g`bm_j+HdSdex7KHgqD!t4hXC{@26ImxC-M-gXk$l*AeUnhib$5wln@Tf1uK>9xG+hXjQA{roG<T*?d7nBB3Iu&MTNS+UkAK7<*;&=YpYo!5OnCQrZ}*|CHojY5)_OLK}Q`($Sl8dEMtzRY4<;-*hhK&B3B>LUysZzI-vpB;@W;`IP}<lcpNv*;Im<1=~`*8U}=cE1R|M%!)qbbM!0p4UbCmqdQLHzaIZ+&w=^K%VDur<9r~^H(qisD8Z0~R90^r(8aF90L8)E38rXu^sj;n{T`e-@)^MtG`z8ymmyc@~v~=A~R<l*3S}KOh{|XyZr@yfTHMVIdYT`ap*XtAsS-9GATJzy(28buCU6CB<_S#s4BQlEGkrE;^M`Ll8I;>)AM$%}ew%ytgOPv5n=juuCSdmzfOLcUODydxma=M99x(ge+3Nixp_0dLlnQ$dA4VYveYhoQB_(zw|w_`Yu6UaU;gD*ZKPdd4L4ivSkJwxlqq41GDLc+$+Cb;RIZjoz<x4AgGd>-%(^fxgBz_rudAw>+AJQTyCew}1B%W+q7K>i?J_1E>Q>wX(I)SR&8Wv=nWTb*w$YBzSp-5L#FCL(_G-uNHFA$Ls#R^~aGFe^u(<-09QK-tV8j`!X@JzpIO3iGN-hvu|sJDi-z^dyuNZ8)?J!0xx8DDwGCavE}fm;Ocs5DBb}lW=2ymr{>{q$HF1-Uf&M(aNH@|G+*fDj2i7rMhfFI=fjuN82D7&d*1<i}tr)nVnHC5@&p~2zP)k7}vk|o!QwAC2}?=G8|7ZtoO<yr*QiD0E$MYo0N_v$?BVjozrlbPx{+;@BYyOxBh&nNA|aAfg}ot;-%qmxHI#IWYGG5j1@?THgOEbaX*ryd<S$v%Gi%|8;tP+aEM%^C^_9cP5oKlkLRZonaTyYbbdO4a@x$`(aS#GdBj3f{X8gA=(7GS2}!>%cDK~tZF-EH-N|+v@x=f9<==~Q;y_Pkfj{H$61ycX0AWr8%52|gt$ux4!_{#YJ@I`jB6e;&42!LE%d=SQAW`wJti&!pR0E9zvFPn?%^HlI?LRW!GRM!pe0KRUE5PS-Q7V%iVccXM105ue!>fxArU=S$e&cf$w}67=#64w7kP7*Eu3~u{^U%g-3uNNKAHX3V6ys7q6TY7T81HgG5kw516<I6JU;gdoFXxO)X0$#C^E@;G$1k60^w5|PN;?!p{1@b2uP%OCFn%=h0~K|d(H1@l&;&z3xcw{yR-s}kvXEas^A&<1N((xZd;50ab9HDF5^!&S7Xr8Mh?j-UZcAgRGk=Z3a53KjCaG+}C1q@R-7`5A5ff$$zZd^NzqEM-VLxPS>a+Q%LN}!Nf3pAhA#}ln9}iLhp?KBg@_AUczV)Ddd_cmc*C-ni_EOJVfM`}aTDqWUzs9<taK6&yj?Ari$vy3^zS?!$a;lE_3*q$M!@0iB9|Lte&W<21X%F&H8`C4Vp14x_Uj+~xjgpQMTDK>oxy`NLArcdGN8<xkkPr838h{p7Szq9^vEh;e!5D`rSEhp%rL;H@P`7R?Pz{mOiigy#xI2VB2%~I&S1Y{z1Du3sVD9ndufz}De)A2I;}@T*lkh3jpB|<Jw_B52W%9^XQ^L5rRYKolYgEP($d#K?#Cqg7&P$?2cR~XiWy!(%*RL-AQM_AjguuOv`*FcXa~!fa7hirBYi&ZJj$`~YYy?b=d$(_Gigz{c9Cm)UUyHG#jK>!0u6XM~m(#OoG2X3}x!vM4!!@}JwWx`)pmYIin#vzN8EWxwLd1l%qNTGqk-;%rn<ZTc<|&Z+h-i>}$|xRBgxN~vdwju!GfqD20ECIfnADd(=*Xp9z4#AOZ?ye_{9XlB+XW3($EkzK?|$ny5EI0TRjSo&S_ifiu*P8BOE~A`4SoOM;E7O&m7?ETH*w17@k0Yp2LUz2U{dfP{LTS0786eOF#oJ+GCm2@B&K_l;rf$#K0#7Z0HvQQjQ*++6POu}yyIdJQesKbEeBL_LRLac5EV>p0Z3_Qv{C4)n*^1ea4i-=s>;_gZ8L_K{|WjkQDpO+5iAlbqKz+JX<^rvA_f7pyCE9S14zi_KTU~)xRektF=>ZIUxD{(c_U58WRT>Ib+_*l<j;+$;a837<!9K^lo6ooq&-Ev9)3?jG$Ktf&4OLBIG+$*|5?*@a0&Sc)L&n~5qR<M@lK;r+2G*6KLazN21**!2}fwBZg_cVDqNstDGeA<K4mCLic#1SN~EYFV%)^|FNB4C<VB~k9T61Vpf(JcZjUq%yD*$l)Cch!^JT<S%upbB|4LuQ<I7)S_u%h@I0M9XZr@Tku|C!*GBvNs5r};KuJ$ezTS6SI)X?o)13!VLy7;>(vFT&&90J=f{4GxSM({cy+a#`9`;|E=mSJexFO+{&t7hR1*7RXdk*^{IF+#Z@<;l^VwhLKNs(v*m7MffctzPLQEM$4a{TsW){iaWR`xY#cGEzFR+d}~<>5u6#R8U8~WP1lGS9{SV*GrE1)m+>p)nlsgr2#=jg&dkR*<v<qWT$)ibY{%7rL8FR>C9MFQ4D9E^<eojs)j&3P*;JX!#wuP1IIyelR5Us9Q`?8EbP22x2u%FZ0^HdIkm68d?inP9CfJT)Th_kd|J$gBkqZiGPT^Y<<dk8|HBH2qeJ~jshQ;(ShKN?9%Us}bj@PRoF{7R8%h2ul{9cc<5x$f%JJ0q!X==(oVNGlaP}%?j7qkG6ht+Oxl&mO)fbjyA3yOPK6>@F`v;F6a-rw|$khV)9K`2FRZ2*Ho%8h3{fDnV>P;yQp**8{ZJyIC=v;zO^NWaCe;#XJ()74r6{PJf0iyElzA6w)>k!GRQczPxpsPWzgJXF-H8{Hc#&9ejrv}Fa$X7(}+#77Pa(Eg$RUJ#9Z<woVu3S1oR_REYbeSZXq~ww&%VGbO))8wHcxuR^74pm1rQaHfymGapZw?iu#fL51r`Y147~(31PFT)6`0L(x^iQMkn!g-?l_Z;u5F2i94LYw)nXRc-Y1F#ld9+dOZo$WCV{Kt&<?8C&n#l@Rm)|>gI2pWHmv&y2G_Tihu434Vvhd}~lR7Cl&I@|nhziR<0zZ>%`I;Lxw|w2G-;S0;<a$7(^{BhLs#+uI8Ib@89be&Kb+dWVPRVNbJz5jh)$j|l@O>$E_iQm-L04O!bSz*{Y#ZjPpozJ)L`Qc8uRzShrQMO;BWI=M3xRsw>q=1BaX9lJ@MISVnrSC%Em5qv?ZFXs))x#{)fTJ1Zs2oi3OP*r%2aW{?j{kc^;Ybh>lP_u3N+!$z}@olq$)%6PaO+{%985wsy3v*T3QT1I`f!f!>4dX1-0j+Y%|HivsFlzlA{6Q_}N-%4GtDZYyn-(*^0xOGRq=b>CS5wu1l(a+BG$X*%pPc%y!)MP%Krwxz~cy`q?&}ddLKHZn)~#B-+Yju?%<lY?K?-=cBE*<R<H^S;J&~k<5Y&X5mnB0$3S<B%QY`sEue@&jjGtdpK4Ers3k~p329e(s;fzBvyX&dM~OiYykjv22CM;O+fssx~a`DQGHFT_xD?uzY?Q+A78-q1=urjj^1p}9w7yinvE)~)NHjPO!clc;)-uP!Xzczh`RQHY=Tsg2@m0xxPe-mR;*cj+F7gCXRCOfa;@nVW%?YBwvkrhl-V*ZSJfGF@M^it_z-!;qyB0Xe{aLB(bge0LcFZ}T`LJqc~_!=Y<xTl$7N%d6loxdZG6`>&BoP9C5eb`UMB!=y>tJy#|MArJ$>}#&8G+I)@h`!e&=)Uzw_|%L4Bb4xT$$a*R5F2q010d1(&f<aM2R&I=KG>VJpL!YXr_Zyy~1ZSfpEVoyfTa+_X2&Ty54m!jWf%`}=%JBUU8`%bKgITfjFtGgXvugSE^o9DV+Bp$QEE0DP8>H2w`>nR1PWI28px_rd*Q6rP!~Ak+1uN8j@vJ$?G-)3;g9))Pe+YMc<I5xonGskTYZG2&%o@5E2MVUYyf`u|XrUJ^f)az>d7Wb?u?)dUGu>r$&1qyCH}Za^_EN#9@P_&8ogm$eDcdMmH3>N3}YHrJvh5y~o*ESCf|*ENglR#PdA<#!w_NK&J&Ug(Awz*fq$gZ{NO^a>_b_o{P+B7(%#cN)z*6_Npb`Jh-u!`kr2lclX$via3a3Z7IF*VHEycuaDb!ii1sCHGj>a#5RS=QOy>Jz8xF%jNoCU!)sfz&q(+#aHi3UZwiPIFs_&d}P+fwoRLBu6r}Kv7g5Io|;|lnnfuB)3Je-@mh2tIv&Wzhgw{U7ujLoYDN*GYzccXj&&`*Z@#Bi%B}j4jPzMaQ~pOMQqBOVe}JNQKOJ-Z<`eGe43RU(NB{hJx<|#Gp6A+A>0X1c($t)ev$+Y;anx5=u%JU5p%oMKc2`WoND^i}mw#w>`K@YIf?fYWj>h!#7jg8FQ;nvub6DGQozESEMZY$_QSe-rxLF0|wZi9efN45fxcXytNVUhQ70=@@hCPoM;dxHqd{;+oxqa(@0W>fo)c',
 'export_snapshot.py': 'c-oCtOK;;g5WXAme_-7MDa2NSpt}fM_)wrJ(4yD|Xto#EAQ*8Jb62mB)Fy6_|K1srlI?gmtqwNDdGec?kDAS9*H2w*JyWb`>rNGZ&VK!Q(OGa{znJD8w{6$EAK7Psj#+r<!FlGbD!?(*;0`Kicg`Gi$|?`lE}zY2(}}60Uwe=H4#thGa@)$>jqjga+e{~=ZEMykPa>8iH~2c8jEdb(S=&A`<xuhUZjM1PH6Nr8WHp^=C^0>yC55O2dC!*93B%`m=Df{aR~cWK2As&4s$9#$j$<FsE-x;x&Oe@ATwJ}wQ#zx8eNmOs?6*Jv_&~-44elTsd35<`{5G5*Z#!rNG)1dXX~lbAF0Ob+nV}u1I^{r*1>f6-ttrF2YL#|^^2u1)T1G2^I9f4NhMk;(cOu*D4_PbIh~a1fZOc|8^7&8pzOSM2H$fsaxB@+R(>5z9bz4Z89h&A!YpIfr;EP4udp-}PHx~5j*7lGLP|IPYWEcL(65)(;v_vu0QP6gd?@8j?-WGtug94HEIpeY6a#a-2dC++f@?-~39Td2Wmd|$0jpmyrvjHv}unK-(qB0`Zw0?+1h*Dvahm3u<3L8<J9acpd2kbAZ?lm@-75uvH{4R|GInq5gN`c*u!{UbAN&}T9JOt%6XS}x+{gtLsl}s-lI}XAEo(NX1TPh!|a0AE6x0YVI{;o1^3s?$N+vgS^Zq=itW+PQvyREu>_kQqu@H%E1DsA+bEe;Lkx_<r)JA^R52QP3LTMrk6W<1f$A-oo$OOVY;ZB?oIPAg{7N2~{RW`N_kCI|RpT~PO-$AR)Y>Adp2;~T06AF9p?Whx#j2BbDa-!G4`IIY!(wpk2r$P0wpMV=ZCpR!0?lt;M4K314s5LuRQ;Ypi&<UA1?CaDYnZ~ilzThtD5SZ!Rbe6byJaK#pC6~-rzYq8#Yqgu>KWE%he2MmtsQdL$65~9qY!ayj-Ch=12^^O!zi$B^$3Pb&U$dkfH(I%;Qbyq=5_m-A9aklJsl!2gbnx`YJ;@?0j5BZ}tm_o1@;djoCXFZWO_A6aNc=tFWC;pG~W5r&vLn4L2HG(WS&p)+Dgn>(SUw2N}IfKUa7Fo#^#;krpl1FkqWBl)i?;Dho&q(zZWv4*_;wU=!Lmt>F9^~sM$a3Efm}DahB^T5`80t-?xk>k!ZTm(qsO3)@$CTe@&4kPU3Po4h?{5XdLi80e_%pIp5$*%Wmw_}BWY~(*?>fL;9vYA+XBa*i2cP;*rVk1n>?NZM<do48f->YIDIfU-tietkZ5R*7dt0S66;PJXFE4u@YE&JrQgMva{Iq@th-u|%JQKp?B8v@hxAQRGaeJWFDfVVfx|@JoP)(STKf)|=$MQA(1o<K1yd1YagfBH&tHzWl4`(<Yfy34hDb4ZF8q2o2{C0nLNCaOoC1wa;UnP^04kTsZ6e;Pxl9DH+;>X8y@-L$=LTC',
 'priority_probe_snapshot.py': 'c-qZ9O_SS3c6V3#KOouzK$*mFM%G5AM5nZ)ajaUcq>{$=rp8k!kOWD1NPq^w`S8@7bIvihRHbsuA%~oH4>@cq|HJ<z@4bHg1vJ2+q)kQT@(_*g*WIsQzkdC`@ZiCN_v?7I@;2MaDxD5C@gni!=`<-zZ<?<+agmmJHu5G#epM!gw@l($Qj}hrdGC^+w@F!*eJ|fsX`aO^?|rgLvUIx+A3S*Q`DgPYUjvYAx-E(%tHSxV+7=14U2pQD@=lip0DGI~tCQ<wx~=k}@5N<=y{?ifnP~u3a$T*`iENl{S1T_4{4*|_RyUiZRF8{a(v5of31*-k)^W8|4|&O*Y@pQW;jLMzNYu}AxvkQbe%wwrMUJ@AMK}6)m97(pzb#e(J_K}@iR=W*e#+CVk3VL~yoeWS#;a8_MU*OB!TWf-s%GgFru_M5-=DmD|MJaiZ|wP-BF&4mx*5PoCduIPvH$sJ-<~{w0X5+D?dP9)@Xvn^E4r!tk>?}U4wlt=)sHtDSn(M9Ik?7>FRtsQ>($er$HVZe{=<WZ^y`V=w;&E)#o1yDG=rdjSq_e0_kUS(n<T3>`EIi)fO-Zmvl7<3NCuLEu<;>QyTi$S{u#{H%aZCUFD{~_DDon@NV8cmE#j+H0#uS*PkSQ)be2?cx`LUORS{r$xXQ1RBIt?N>D)`pG%Kq(n<n_a?=h6H;LbxRO^ATY@csK&C)l5aj{k}Ts5bNbvRsM5_#XuNIL|Uz)Kj2|d|SCXn2H)vC8~T_H}7Ceb@YKduGvAH-2}9y8JqLG@aQ@4vv{4rp3PTVgfHgb_xuG^oXv_fo9Esfnj|wU6~DYH_o&Hb;`dyGm}O-hlQ6J#4tqW=84Cc=7!ZudpRV!}fb)A8jCYk*OPB|Cwt=}PMvHdzLc>UaGI}?F0<fgIHzZN}gI20zmFxv_bStqZH_LPhEIi2;Nre16U?$!7Ci(0}b|5)7Fog@bNk-l!E%rqpe!!HuPMB87x(s@Eq9s*^b_nfkbT~Zv(tGIPpL2V&P4oOBO#mmWJf1=QZ{k&%DBJ`vz<>tiJ@cL%esu(7H}fX`ET3-Icsz*agn9z2@q+)baxn2%x!_Hq=tb8?8nIq)R`Jw;W>F%s*%h!i$zPHE>DN_pGqQWHiX?#<VYLt&1Gh=Dg_w?A^Ac!avck%~=ZDDP18f{#;}H}12M5^Z-~ich&^ycWDnN26;q)7g%4v~qsvga?_NKa7CB1XlPa$@_32MF8QIELhks!JckA}Lwpg}MAK3;7TVt0Kpv4jTt7SE`n!3JHuoEF=}qEShCA@GO5pUwyCv`QCoCD?<nn6ozq^eCeB?OtdhpH*qKO1}PSTUK7|t#hQ1HqxL5o1_3rUc;(*bh^Ni=v^j7I_JZzvoZG;R~2+4xWLcbv`A|7WzD~*w_gFLcTz5+5+ng=9n6zzy2OKO3z&>xM`mi9gJARmm|R={n_MQ$J58px0eQ%~T0*h6LGDsq;*`EUJ$>sjcsS|+qe8?&kWlS1&9}I-x#Lg{#M+p^4>S*ujd<7L-_1?X>w73-1nMm}U^u|*Dj1F6AtMhDfeMaYS-bCD^aMK*X9uDG23LNX6czrBx78B=Rt2nUlw_B_<S`(<T+<LiM2goqR2Ua_WRHidHO47gHHVKtdN!Iu;{Fmfvt*^PhRkCnaVKXiP~M7Kew2s`x0@L(PEg02vl5GGeedDJTmL&)>cMj)6_AEB-Q3kc$mL=X{8+2RSdx__(rT3mWvY%qU~RmJJ~br+m$b;q4V)UtLvXbvX~?6);2b5B2}A80EsZ^N6{l4MaA9`=`_;0Z+DpkLpaT{daJWjd1nvs3Th|-4vr6b0)=3_@K%CKhH4CWOnGZ6yf8MA47L~BMvTAiRM*h{Kjm?dgd0AO4c)nSF1)%X*4%azw<~&QMk^*qx@Nykr2S>1ya#-FM`t%2Wp0C4gmLi7=hJ7!5+*30PGdK#OxFVj3ljwWHo;ScgwGd04eBp(M-q&M=0#V7pG>@Dx7y(j2OaK;kj%U#^u$5VUl^smt=|w)D(<zY5O5k1VIL(m9u#n<SwzGv2Y7FLX9OqU?v~s|6^K4d9XGOjRX&sCE-Xop^;jUaI$%bC^tYbr*BbuAiIDinQc`<WPLK))?5w4xv%4D&9kL9L5PXRVo-)8n|LgfY}ZMZSzRXL`K^*zyFG)s%IFMgN4!M5KoW5A|I7i1m^?E_fa7^THDUSvQ%>9nNt8@W5c*Z^M!f=MQ@do;_QB#Sg_r6{$&W3KoGPe>vS$0k{$L<SrR1)Q@$NMlKN+iHp&T?(J$vvYia`N7=j$%Py^kfCeD0N>3NfM}i1@C6|am^Vy*N+;yJhEOj7e1e!y9YsR1K!RyRcU0fmFR#U|kG<f9aFb*+Aj@|6fVBu#AMP`Ggfd1X)E3-{WSp#gz#AfAiZ}MhvG;vAw=GN}#YD(3fhmBb{-iEq(F|eluHqXQ)oZwN+3y?37&VdE6ki?CCB2{Vg3`{FSP=C#IV>EoVu8V46C_KLlHQK-FA0uq-8XxbGK6&ogM=VP04Me=0U;QO@b<3P0yIv`8j=vnk_{_VA8L-v?@HKMhdsww0m=29Y4Y7s$3Ps&2z+@#jU)t=f!<l{7I3+YK_~(aGf80liAT)UcVYcf(DN-(@<>uRidSM@P{zNg46B$N;jRf%EsWgs=OO5UU;~Q7&@l*RH?2I(S`IjLLEYn|=WcpMNq4K|R&k*<>6AHBjyG<Fhr=P?Px+7g5ZLD*?U=E-dp;T@pH*XZW=Yx@%OC9<E@<c#n3ztadZF*Vc`vSTpi;$VEeO_219jY0hyo<zBgMm5>e|jmkB7tazNfSrpjnB_a*VoIp^S?nR3Jsseek6<edArqKNpdupnzVr5fTe5uyh+dF`Cj!W4{k1iEA0A#v&x{Qaa&@wp;lgka#S#0$9~h6ETo`swF!or+N|7l27h7bdJbTp}Zpb<+CQ2#Md7!p1bj~``+B%ny1Z;g(^-Dk&vJj6s~`0LrAl>8iFTUgfECw(vm9iNv>jKj@7STE9S+PA{|LhR`14^WYP`szD4lg_nfk}o!=*&Iuj`G^`20jabXj_RxmW^8Gvo4WB>|}<1#@bKnB;H^fGyYa;$x0Ze>8rs@g>2SQyLGK^?CQV~om)PQ~U9ZUjrDRHQ<cLFT5c8X#xG!&~YZ@i*#d64Htj{s-~A5|4HL7d#g+9O*7e2Cx;Oh7d6KmK23haG;Lko_FBQAKXd_Hifmli*6-~XXyfJ31#G6xcKEk6TGY%SNR%*@ySh<lmT^wDrkT%`|p$KEJPz5ptU5ReQ%BIaXjqpS{(^xEPsSx(A(r?dQBG*C@h!p(c|BaT|;jJUgvCL*=b$A96srLUmkr$mw2F0@0oi2%=_(d=;YfDpxq2`)Vh)XDqQhCZ0X~;zy!=d7qxqb#qC_XjJH69TVQPId5?K>obu-IQMrM~aRdVDSc+v`d+wv?rW<mdHz*|8<x16Ktgb<5kA`r4cr-l1|2@M0{Vo3QiBQ#B$ap0=gKLRN)dNU@#bWEPmRcFgnTBxQ>yp&@9<N39TOWC|bs?y7c$A&Ov$3x5Li})XBYkO>jU9EIWU7?<cm#A=r5e~q!RV;Ag&)B)s^=si2TFhA{T`G`q;4T4Ih|eNo`7qUg8U*)GNj}eIvb&^Ny~7r+cV>y;Lwhf<OoPA;%UX)Qr0tf49W=dA}t6-K%wo@D(EYDzGXz47+nv#B^SL4V9+GX>R1(-a!U(a#AVU-cZu<am^2sEcFR;2X>$**wx$zO=F}?t&J(gtY@w!7W|{j$1_~0~@dN*<wh8Huktkhbfd#2`4<^bFUhFP53vf||Bmw`tjaM*4EbjGr6O4WC4u!+oR^w*#RlJBKKL|t*wW>9SkJGH{ojQ~ktjJrXAkD}@w5ooZ_f31wJT0ga%9EvjsXQE_q^SX<q^;)aWIAjrR7PERHmFjnI3l5TiIm#KH}?V=2CRvyop{gr`CH(e8`^G)G{rpU{%!Mqd^hrLq2SK9XK#asGc_%jP91OUVmofV)RICOue4G<%5bh=b0iUZO4(we(uq4H&=9LXkXyi3>5DS0)Grep-Ig@8^X)2<qOF#hNC_g~15K3q7PQN#!QZKu$7?#+i18K3;Wu8f6|Ny{{skq~(e*V75l@x5b(u^rHgNjF;Q;G(<FT$;E9k{wXbe&!M~sB**McagLQ%yiR~8%Dc#N>{5JN`F^#PauItFtPV1SSamhrTV<}f(RawayVw$lfoG~`J#pf%dZCwPMgqD)IP!NK7+2hk6YZ-oh~vrz8gbzEGKV+W4#!1E;rBYSK-r5T#Ws~b{0iryoDPhvzN@qo7>v4T*3c2){P0Zg*i{am{iaXpt2!q+PfVD!VCLarEjNK>vE8{<pTN=mv$OT8d#(Z*&g<oJXw=tu_Z_1}zZ7e_g5d2{SJ`DWJ=!RZ1=%j_vCHGoXB<ciisFx9-c2~EDvyNAZ4X1jt}27y>4jSUbZR{13GA2J2qlW^Fxq3j$=k)UA(*6kt%*%I-IwxhFiZT%I?XDp=AJKtk9t;=28%0sM0!Hq(90LSvu5{p4evDXCG@QSI6sLFQMqz~-UrVc(!g}4Ubdy*H45~S!Jk&HElzHMmQpI~0TIQZ_}D{q~a;Hx@aTJ%vL70dE83PVyreq&Z^%788ji<uM);Z{H9#epu{!=4b&Kk4G&Jx}sn%_`NEhwp{}sm^Z=%F#fOrGl$Yj4CFu5lS9)|MdV+f|fzU3%jobULe;9wvDwe#AQiylEb_MJqXkaO?=YA1D{Odb*+0gXoz<qx@QJ;^95O{ta)lry006?4%3~%yd@)rn3xn7YX5vcbS&d?c~GXmB(3CGyY!hv)VOn#RnPWP{1kNb8}F^~oB|miOtVt4BaMmRpPE1Qa=0-TS2$TEIII$2yQMb+e!U`x2ws2lmPp=~;3}m|lYww#88z;+hLWGE^oftVv;uRw;-;$^vQw(ta^15#rY1FHBr9jn=qb=*=%gCFK5~P#X{W%^e1Uo$YkLxaox9VzVaEkpJOPL})AonJdtHvFWv5GqCi<&as`=i!uba+|pa`kX9O_)b{8!@;a_GDGgRKqae)r-RHM}A3kbv6wM^GaQ<`qKQRbwt17RpX~B_)6PSRC)*I6IZxV)5&0Z3(_pwcXs8A^pE8lDM}$we5QW?mpiA-Fn}y-EF^Z-0~gwxVluoQwpoy<P1kYVFZzPHijfND-bZp2Gxtfn>hg!&k{XJTwH~ZhE2W+$RiL9s6AU6_$ydMK17#e9%4$dE#x3u)@xG=bSiotK9oLOLhO!EjXO4PRc$l_PVOT7$nE#lrBrVRX;F&+Fn>@dq5~KGILOi3ihP6F*V5EV+WRXf^xdXIu#i_sa<7y3Keae1YFGGrBrGjh!r}_rckE?oSF?!wXkT<^D^K|Pr=mNs@lK5iXiCs{i#CqOefC&643)*H96z$0C7lEyol=6MA+-GRaoCf-l;guESOi|K5nLO`xL5mbKT5a9rEnS)fr6!K-c;<stTHAtt_7yn3)D0SrQAsAXH30}70E=}wacFL%K$A8Zq!Q*gCKrJ_xgZwrSY@l0{w%iZ<-2ArGCL(g>nUUmMqW~<m)lviOYzkL-GiRg9A0?G?aq#l@eJnN)ZGW)c93288L&gH#Da3u;%Z&V+I^>B&NV#XL16fBYGp}Voo8bd99dEx4@{>uywgROob;tUE=t4Wj<-eDf1)Q&VNLWM)!QQ-=$WA;W3?}(IOk*3f-_(&74aUy|h<o4#My9nK$h0_yyH+pbopul0{rcAMYB|Q%J-!5vlUqI>E|D&ukyX=5S<1jf6TmA~@_+jY;}1>x1wcQ4+8#8Zr-d6IJ{}{=gF}us1Yz09U!5LsbZ;2D;~fB9DVsg=7G6PEthF?un=XjZokVB2tD%k?WxHxq4T7S)$>1zNR<JPuJChyr7Oe<>~92UeM^D{8#aV_HDdEKbA`_Mn`A!bkWwA;FxXtQ&)d7)NmxUjFL~GR42j!M#`C9x|f#PzZ7V}Co+MeHor)Ja|2<Tq$s{ALna4k2l06o#zXK6`KXn=Qh=2a&H`(U0jV~ZIzsDeqLq600D&x7Qf%iwd&@vP0^;#X+*`fdE}?gfuCInBqnL!Hf;6&wX=Cp#GFjT3E<No1b*Z-IQmRSilMtQv)uX!C=vY~HI@PES1E@3sS#n_~3pa7BR)v3^jkp0X(gF-}8wzLBf&2m82wjWFKuoiSJGjZhP-}FPtR*5<Oot#btTTju?wmP_pQ2gx6Brw)zPS>=@!sHBHl@>sCjdG(Slb0mF2R5q(R@Un7%X)#5`llgIssSRlGpn(Wq%Fy)}UB2`yQ(CL&InUPF#`K>!*qxx`)GZP5iYVE%DZ{+B9~Bl6M^huHM0UdsS-+TqCOjGU&~=>rDy%KfJ&rM222&;I*)yxKQjYU-<x0AwQk(oHW@1L@S3~pieQqMQK*R0^n#{ICOh=V5iPX@BoeCa+;>%zJR`Bt8KY7ZSKZZ2!&@sbsTMnYBcTtzRe=M9|&_b%G!Kc5XYgBC<Ja|eMiO-629cMiT{!^4#RNGupLfa0&ExGgpm0IoFM|eHTr@mMf@xMRD@b8t<kxu&fzR40CH&4x#$a<o7qqT9zruFTLSuG%uP{y3Opv*O)#o43Q_Vm325MEuw79(t^hNn`>!Cc-dWp;+&G79MMA?&+&oL}1xFoS=lV7+wN-PlQ(!P8?64RI#)<;%Snv+INYJSg)RF8sAVVbSx|0oJ%7FsqtVTlj5J-MuXTi~NEZ*32*zODuFy(U?QtUyA#2!eIko8U4C(%zcy6uoM3gz3ZjOVa2ke37WDZV4Igbu$U`LWwR*lcdLFudPsT)uR$it1tg+L6+>QSJwyL10gCqQgyeA=wURx6q8JhGvUJsUey5;W(lznQi|xYmlk4Ibi9vW>mfN9njUKPh1G(1k90H*Hg=ChTMyB?5pe#j-(-U-;JoYc*$DE8alYNi4j>HS4k4KW04d9@~o4MaCF=30>Y6q1Ei~8^j32cXRU_CM+D*O)Ibop9PvJ`?OSfJW0RYUV)`gy+9e4%V%8HV4IAsIKnn&}K-nS6hVaNYci5E>T&;8AKxEpAG9b*;0y7@O(`Cem23%Moj-wzZ03-ExC_Ds$z2Cii^TYd-chTv|`_t$TZ(f{Ox&|B!`pDrpnpl>%h6(B|@q%6mjtNILG2Ma!CdEYWBueo#+2h8cE(X+zh--J-Z>7!NCI2+-x;2V2`DZ)Bbb3qd$v)k~p(Ryu#oPn^j(}|?*FL2=8Cv64tAnde497*wWP?*xcaRR<e5p?Jf`$v{l<O78#Tt4C5OkT4hgn3h26TXsTpCXO@UuX_#ne16U*y7B7_P5d%5nyc4pOS9-0({wz@fLch??iV{E}0gh~rNkoVuM&e&~&bt0>1d>Pp_jl>h_@KPZco0YW(Q`4teQ@w+=!&hL{O{)O=lEqQY4v?9`LV44WgiJ%ryS`ThUGBInd6VRW`*vIF!2+;~9V!&XUz}-u@{BTMSacrRNO0;<)kv!V%9wngSMb?0&5^Oa`YF4AAemN6#FA0RmQ)RddBq;4dMf<mRHay2xx}u^reAOguK^2m=Wr0oIDDJANXM@9Yebebl4VoY>r95kji68`mWSrEFnLQ*8hZrFJ>K)+@j&rf*-u^qz#Av&@JI`Egt86eyl=;BSm<$N;2rGc4G9I}zwn=2v0|d>^WSvG_L+hYK8mM>zC>~8#adzQL5v(SgsvG2}^)SfdOANHbK+vnWz<eeSdvk3vx{1_+YNeC|-g!G-gzTl58BG4P_{+a%OK-Nl`Ss8LSa}!Azx*fYhgQG->ECX=|Nf`{`Hx@!{I63l`}NQNoI)LVGx^KEq!6g`>!1Fs3Tt`az?FA~niRrmlR!zKjQ3iwCl|m96gLvNc|hevrEnF(D{m4B<vT^5BKTWcuaV-!Aw7dW1O!Q=l>F5G&VCM4$XZpLLPc)`2Y`y&Fd%r!CxnA$WLRZim7~B~Pb6mq5e3xHCS5x#7;C0$iMw159VDjSD^VDY9czmmR^Cm%^&tI=x7>nnw5nmV8?y6ZDdvhed^~K1&omd0_2^nrB^Ilur+SE1P@TB3@0lrg8Zm_wp<Bv3C@?RG05GD-Rr*x{Oa(T$8eb(7rk--U!B*immF<5>CNJpS4=@<T6+4qcxHH$tj5D%Vzblyy&s7?uoH<r%Hm#7vJc`X|an_J?Oe(RPOk@Y<L$eJrACrYSs{q7BASn+msAjdMz+%8|8!}5K^+j}Nd+1_nM(R`%;_Vztoq1e=gbOM@NKOcs(eBAubP`DVHYvCVsj(ofZqVdtyC>W-Q^7MuYAT4sNo9zvoe-cAUc(`)JZTQ$5(2MvyvZ8$V7g05p{K!hr?X8oUC+RZ#B$v?SoIV<q=Zm)CZs1SIOhlak<RRMM@-L@G|nmcm_u&wBtL!9>GSy95bJfH?4iuiXl>ZP8a&b9jsj9#H6J|jce!q<?ft>7Tc6UY$X@zS6}OJgZfep>kD&bETHMy+TjpkW;U-%c6XwZb(;nvQ!-IEy0m{xmJd~tz?TgT#D8H!d&_PC$tPMPhjDZBW<qhDymTgpb;UjJVPX>DK=UJ!IPCxa?MgSf~f^3|G_#uvT4v(=Nk&$Ya()_`}!{avL-wsK1B)XCig?vmg`S`2LLH_p6?Et?sh*vDVt<FO~s-J}Zw$qklhUq>KI?UFKj1jr}plQ8gMJ@bi#P_l;Ok%@U?46I=quOgV<@L_h$w!7~D#`u-QEaA>aHu<&QurOZ6`&ua**8)<%1plPdb7^^Q|nc+?XJ}H5Zf7eWM%$nG=F^W@FhqPVk98t?1A={SLZ%CBO1ysh%kX5ORykQ0wwu%1a=(u&{q~NC_)E(02adno@KUs!&;;jU3vtVs(`dpG6dYV`N{Rc8qL*_E!5pZSG!*f7`BrEV7hI4_?ILa&ck*AEYFyRL}FmrefpBP6_dT7+~T<s0ELgGhQh^tazU4O;a-o{PRHa3mu<)&UkLQMkalZWHoQmVDRr49SMYE|56nA9-r*fhTw|baHo+vV4i-z(Fq%1yH_&*1m&GP@HjdCPFeaGJO=OYhpwN;itmj1y16&^?JnR?DEnHfj@1MVkUVQiFSR^%4moWxquov0w4j5JnQNh8?#qf{6{qA{m{O$99@NGjttJ)sbMJg?*u%-R}+c&TOK2p8*+}+AO4niSHPLa@kp`!qDF3Y3Z*yAtJ-4#BK!uC4vm|B4VM2TSGdmaUR9R>Cpo;vU;j7f~&CwK)K)oYdV)x(U!r4?7+^jn5O;e{H#C0m^0$g=`Go9KM0=W|5=4l&XM{q?68s1V2a>+?Iys%QeV>?JE4a_v~a5oyT?GYDrQa!QTxKCve%GDIL+`m_n~J%<4p%lV@N3cczpCggzv<#BZ}I1GnjnTJD|XTW7pG#FOD{gU!?=@Q@Er?UB^NWsfzaMXmtuZ|kP%oh!qSy?^;ND>Y{0*mVNyG`+>^%9=(tqo!^S>sp^Wi@y{R4}eDCalOYhpS45O>lq2I}rK-Cl=IeRjh&kaTcSk#;zyI4jq8!={0EHq4FbDnTqs`|A-l{3#@Xas!-8U6C9XjK^(H`ilQR}`=X$M^(i{nm@TouXh)BYRvU2GgcM6@kvFKwel;-AdS@Os(txjM)H29RcA+g+(rP8@Q#6%o+W63fEBlc6x>0!i;?=8wfz$&(g&fRTKOG==R8pjFLg78g_2hK*WE_(=+1HtJYY_Qsg<e7eYfddyXsiu3#<aCluy#_MBYkiwgzq6tfl(osMW1BMj?;KSO?-D=DR?hU3skMum-L-rRBUkngU$HW%%6HZk^|MTw*_$+#61)T!8sg!BajBg_dj@^y@o$l0{f;$CekU8OlrwCodsmW8#+c@ZfAJ|MOf&&^Oe#W0IxgK7uyKn6VQbNok*g{DHNBIQGv)mTj96Eup?)nrk*uc0UQmodWt&uE(GkrY%Ma7k**l<!KHog#mVnpK7Sp3^X|>-)05XP#*hd+!$5~ZxS;~xUmZu!U%eXp$G#Gw%=m5e8srl*fap-Vhm^_#)ahs}T_E%qztHzE;K_X98DPq4h9Oi&&9^V#o=`d5h0OBzr!U@oce+O^VU#AN$_I6Cur6a?s!`~oMH&F{<(oa=0&2$w();z~#ExYYewcjBvp6+NkhQ?b@Y@rhas-K&Vx9fiZw{EeETK{hHNqPqPXy{SI!$EvaGOD-5PBl9vxC4s3*7wLy&brLV0);@++$x@n*sSb5fj~o`O+tNfOAw}iG~~2=)6IjCS9Hp8}x;C0a?l5=buqS1z~t~Ho_90gOSwE95=s}6&a3oIGu6p^A_AFme<@GXT%mU&V#bZ`+eELdFvM0JEwN*mg?Lodw0{=S?wF<(XcHx9mF3pLXpn^n401O*mdC_%kt5W<-_2|_h0k?yh5u@IIS_~y1u4n@;wq!F{*3pDJ19Q;86I8z=aa@8-7Q4x}-BHLGW`&E?q~O(U#zguD$j;r{i0arzSZhTHIuAfymevWsYh!`2I{K2VK_BV#n6wN7{Sj!Z#RX`G89Yc(Yu?+Ed8tLkaf0Zzx6Tmnp|3?VKEK#6$D;B?T&xQ24w{22(uE+0vQgrGum4@eaAf;Og5U)wJ|w8(EsyNV4ns_gCHutFx`VRbL#ukJh&LNiVAUV-p_-tajtoq2398$waDl?NINBX2c3dNAuA;0ozG&qavoY#d?fE++9J__fd&BTmz&RfbtONb{`PXL;9TARDA*Uy?Er&v3tT;!wdygz7cOEt#ic?JT)UJ3Iwn_F=!KIPxyMkmA|7dd}tDBdN4~a2jy}-*plbpjz;1emC$CgU4YBNVz5a!I~xg0`2aJRj1P~V4EtXVhlXdAUQmPT+hAekfmnUqi;XP#=xt^wd5Ez?|BsIRdjJRKR>IjZ$*+y!S>)vaB3?u?9XdM`SgmUK(GC_+6KH*b#$`W&Qu#ShSrgqcrtv18q-5js7dc3aAQvL)QT<Mila^H*SQN<p>eD~^(s1U#KZ%Z?AAfriy?uWAt@3)neZYoQ+_<5n+=wreIuDT><mt;loPfL`I2%ZHzWi#CAoH=LaBw17pk-l$QjapdvfRwHP6BhQaI!&3I=Z-m4<r$fK&y+;7LQEH>-gQt^V5^)^_%F2m#<&E`QffX6JtdaLW8OtI>j&?+b`hhjQOwp-g7zTzfLEdkMOB!_)JKr*CeCF#5FI}-Hg}})g&{%wGpkV>Tq?*xg-VRrN(r0{GMyp*UXMH?{>ap6Iu?MTxcY|A=Q8<sHAH|T!Yi8`6hNTfXjF^@m9wYEY+4BO=WcTHiNtWg;ZcTJPddq<`K}5x|$~ddIKvsjStm^V<YVGRk~PK^<(g%q&uyu|6aK1New+Ix8;YFN*SGtI6!4RdY>y}9BM5LGo0K6I#kL<*5v2lKB?95Y7W%xHbxG&`g`2`5mJIQP?LXlAc~)%kI&aUK@y^#9Z-?)bodat&hY@K!SnAg45~Kbqj$4Zm=uwo!X09jnM2rbEU4OqIu4cPLiY!&?rrP;@R8K^oF13{y0x%mLq>*OP6FbiNF5W|XQ+)`*)>>4qfbBbdr^!)+f!7HU{JK@l8MEAv~~spWeF)#eUPg8WhX`EZof&%U+nxzZkgIlpY%b(we89{KEx77@};ACVk3ijJ%?=t5)6Ed8iWn}Sd1q{S;f%AH;$fqLWU^G7r8{i9k2C-F2&F|cx#8v5kGN<q@c1N)w0j16i1MLrZXr_h9fj`A~n;{D5hZ~u_zm%zA6Z^ew(t>*R==wnrVa^i4??v^{JSBEsMZ-SAH#9C1g>T^;9D>hq$zx7fN5Y+y4{(u#Y(AU};J1^Je$l;jN+ViHOkJH{C$n7dyLBbL{gq#AS+~bH8a()w(`ex$9GvI@WTph=Nw~+DAKTF+OKVLd{LzfeQ8Un#L!)*aL*~uG2TUbdW9i)u5Y-NJLMuhNb(<TFSM=;WKA&I=mR|_~28EKGXQLF@6M^(`K39g6?&G1*Eoe<(N^UuqdLR#^^G{uet&Lb$k)`P@;W6qS5`5p}1N*(p$&@QTh`*w~)@2QqPWKWk;!d2;VzFN}KbcnIoko$ZfAQDK!EVsqEV1rW*JBU($DdLj'}
EMBEDDED_SHA256 = {'audit_snapshot.py': 'cc231460191696a2cbf692742370e267e937e5cca704b9290289c5d68545a975',
 'corpus_runner_snapshot.py': '89cf31d3b96523e6aaf7e13488c3da4cc819d62f0c0ab67690726d9d90274b34',
 'crawler_snapshot.py': '80bf08dd7a10b59ca44b02353f1cd6add14da26e7a497a011d34681dc367d5e1',
 'diagnostics_snapshot.py': 'c82968c6617017aba772a755218df07e84a7ef2b550f0c588b2eac3933ed159e',
 'export_snapshot.py': 'ac461b827b046ed7ea8fc4488a85d62392e4a95920ca9be7e8d714d3737a4023',
 'priority_probe_snapshot.py': '99547020075067df140b464a69540b8fef2a359329d100d336d14ff33f7b9c52'}


RUN_DIR.mkdir(parents=True, exist_ok=True)
for name, encoded in EMBEDDED_FILES.items():
    target = RUN_DIR / name
    if not target.exists():
        body = zlib.decompress(base64.b85decode(encoded))
        if hashlib.sha256(body).hexdigest() != EMBEDDED_SHA256[name]:
            raise ValueError(f'Snapshot bị hỏng: {name}')
        target.write_bytes(body)
CRAWLER = RUN_DIR / 'crawler_snapshot.py'
CORPUS_RUNNER = RUN_DIR / 'corpus_runner_snapshot.py'
AUDITOR = RUN_DIR / 'audit_snapshot.py'
EXPORTER = RUN_DIR / 'export_snapshot.py'
# Diagnostics chỉ đọc output, có thể cập nhật riêng mà không đổi crawler/hash resume.
diagnostic_body = zlib.decompress(base64.b85decode(EMBEDDED_FILES['diagnostics_snapshot.py']))
if hashlib.sha256(diagnostic_body).hexdigest() != EMBEDDED_SHA256['diagnostics_snapshot.py']:
    raise ValueError('Snapshot diagnostics bị hỏng.')
DIAGNOSTICS = RUN_DIR / 'diagnostics_snapshot.py'
DIAGNOSTICS.write_bytes(diagnostic_body)
diagnostics_archive = WORKDIR / f'{RUN_DIR.name}_diagnostics.zip'
from contextlib import contextmanager
import traceback
LOG_DIR = RUN_DIR / 'logs'
LOG_DIR.mkdir(exist_ok=True)

def refresh_diagnostics(strict=False):
    try:
        result = subprocess.run(
            [sys.executable, '-u', str(DIAGNOSTICS), '--out', str(RUN_DIR), '--archive', str(diagnostics_archive)],
            check=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
            text=True, encoding='utf-8', errors='replace',
        )
        with (LOG_DIR / 'diagnostics.log').open('a', encoding='utf-8') as stream:
            stream.write(result.stdout)
    except (OSError, subprocess.CalledProcessError) as exc:
        with (LOG_DIR / 'diagnostics.log').open('a', encoding='utf-8') as stream:
            stream.write(str(exc) + '\n' + str(getattr(exc, 'output', '') or '') + '\n')
        if strict:
            raise
        print(f'Tạo ZIP chẩn đoán thất bại: {exc}', flush=True)

@contextmanager
def diagnostic_step(name):
    try:
        yield
    except BaseException:
        with (LOG_DIR / 'notebook_errors.log').open('a', encoding='utf-8') as stream:
            stream.write(f'Bước: {name}\n' + traceback.format_exc() + '\n')
        raise
    finally:
        refresh_diagnostics()
if (RUN_DIR / 'crawl_config.json').exists():
    saved_config = json.loads((RUN_DIR / 'crawl_config.json').read_text(encoding='utf-8'))
    if hashlib.sha256(CRAWLER.read_bytes()).hexdigest() != saved_config['script_sha256']:
        raise ValueError('Snapshot không khớp hash của run cũ; không được ép resume.')
if not requirements_path.exists():
    requirements_path.write_text(''.join(f'{k}=={v}\n' for k, v in versions.items()), encoding='utf-8')
print('Đã chuẩn bị snapshot và ghi phiên bản dependencies.')
refresh_diagnostics()


## 4. Tải hoặc đọc danh sách URL

Nguồn được cố định theo revision và kiểm tra SHA-256. Chỉ đọc corpus URL;
không dùng query làm nội dung hay nhãn. File Input có thể ở vị trí khác nhưng
được copy vào đường dẫn ổn định trong `/kaggle/working` để dùng lại khi resume.


In [ ]:
with diagnostic_step('source'):
    import requests

    def sha256_file(path):
        digest = hashlib.sha256()
        with Path(path).open('rb') as stream:
            for block in iter(lambda: stream.read(1024 * 1024), b''):
                digest.update(block)
        return digest.hexdigest()

    SOURCE_PATH.parent.mkdir(parents=True, exist_ok=True)
    if SOURCE_PARQUET:
        supplied = Path(SOURCE_PARQUET)
        if not supplied.is_file():
            raise FileNotFoundError(supplied)
        if supplied.resolve() != SOURCE_PATH.resolve():
            shutil.copyfile(supplied, SOURCE_PATH)
    elif not SOURCE_PATH.exists():
        temporary = SOURCE_PATH.with_suffix('.download')
        try:
            with requests.get(SOURCE_URL, stream=True, timeout=(15, 90)) as response:
                response.raise_for_status()
                with temporary.open('wb') as stream:
                    for block in response.iter_content(1024 * 1024):
                        if block:
                            stream.write(block)
            if sha256_file(temporary) != SOURCE_SHA256:
                raise ValueError('Hash file tải về không khớp nguồn đã chốt.')
            temporary.replace(SOURCE_PATH)
        finally:
            temporary.unlink(missing_ok=True)
    if sha256_file(SOURCE_PATH) != SOURCE_SHA256:
        raise ValueError('Parquet không khớp nguồn đã chốt. Kiểm tra file Input.')
    print(f'Nguồn hợp lệ: {SOURCE_PATH} ({SOURCE_PATH.stat().st_size:,} byte)')


## 5. Chọn ID và tạo manifest

Nếu khoảng chọn có ID không tồn tại trong nguồn, notebook báo số lượng thiếu,
rồi lấy đúng các hàng thực sự có trong khoảng; không tạo ID hoặc URL thay thế.
Bước chuẩn bị hiện quét toàn file để kiểm tra nguồn và thống kê hostname, nên có thể mất một lúc.


In [ ]:
with diagnostic_step('prepare'):
    import pyarrow.parquet as pq
    from collections import Counter
    from urllib.parse import urlsplit
    from IPython.display import display
    from datetime import datetime, timezone
    import time
    import uuid

    selected = pq.read_table(SOURCE_PATH, filters=[('id', '>=', ID_FROM), ('id', '<=', ID_TO)]).to_pylist()
    selected.sort(key=lambda row: row['id'])
    if not selected:
        raise ValueError('Không có URL nào trong khoảng ID đã chọn.')
    if len({row['id'] for row in selected}) != len(selected):
        raise ValueError('Nguồn có ID trùng trong khoảng chọn.')
    missing_count = ID_TO - ID_FROM + 1 - len(selected)
    print(f'Chọn {len(selected):,} URL; có {missing_count:,} ID không tồn tại trong khoảng.')
    print('Phân bố hostname:', dict(Counter(urlsplit(row['url']).hostname for row in selected)))
    display(selected[:5])

    LOG_DIR = RUN_DIR / 'logs'
    LOG_DIR.mkdir(exist_ok=True)

    def log_event(name, event):
        with (LOG_DIR / name).open('a', encoding='utf-8') as stream:
            stream.write(json.dumps(event, ensure_ascii=False) + '\n')

    def run_script(script, *arguments):
        # Output hiển thị trực tiếp để theo dõi tiến độ; không dùng shell.
        invocation_id = uuid.uuid4().hex
        started = time.monotonic()
        operation = str(arguments[0]) if script in (CRAWLER, CORPUS_RUNNER) else Path(script).stem
        log_event('invocations.jsonl', {
            'event': 'start', 'invocation_id': invocation_id, 'operation': operation,
            'at': datetime.now(timezone.utc).isoformat(), 'arguments': list(map(str, arguments)),
        })
        process = subprocess.Popen(
            [sys.executable, '-u', str(script), *map(str, arguments)],
            cwd=str(WORKDIR), stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
            text=True, encoding='utf-8', errors='replace', bufsize=1,
        )
        returncode = None
        try:
            with (LOG_DIR / (Path(script).stem + '.log')).open('a', encoding='utf-8') as log:
                for line in process.stdout:
                    print(line, end='', flush=True)
                    log.write(line)
                    log.flush()
                    if operation == 'crawl':
                        try:
                            progress = json.loads(line)
                        except ValueError:
                            progress = {}
                        if isinstance(progress, dict) and 'completed_this_invocation' in progress:
                            log_event('progress.jsonl', {
                                **progress, 'invocation_id': invocation_id,
                                'at': datetime.now(timezone.utc).isoformat(),
                                'elapsed_seconds': time.monotonic() - started,
                            })
                            refresh_diagnostics()
            returncode = process.wait()
        except BaseException:
            process.terminate()
            try:
                process.wait(timeout=10)
            except subprocess.TimeoutExpired:
                process.kill()
                process.wait()
            raise
        finally:
            process.stdout.close()
            log_event('invocations.jsonl', {
                'event': 'end', 'invocation_id': invocation_id, 'operation': operation,
                'at': datetime.now(timezone.utc).isoformat(),
                'elapsed_seconds': time.monotonic() - started,
                'returncode': process.returncode,
            })
            refresh_diagnostics()
        if returncode:
            raise subprocess.CalledProcessError(returncode, process.args)

    manifest_path = RUN_DIR / 'manifest.json'
    if not manifest_path.exists():
        run_script(CRAWLER, 'prepare', '--source', SOURCE_PATH, '--out', RUN_DIR,
                   '--size', len(selected), '--id-start', ID_FROM, '--id-stop', ID_TO + 1)
    else:
        manifest = json.loads(manifest_path.read_text(encoding='utf-8'))
        if manifest['source_sha256'] != SOURCE_SHA256 or manifest['selected'] != len(selected):
            raise ValueError('Manifest không khớp nguồn hoặc số URL đã chọn.')
        if Path(manifest['source']).resolve() != SOURCE_PATH.resolve():
            raise ValueError('Đường dẫn nguồn của run cũ khác notebook này; dùng run mới.')
        sample_bytes = (RUN_DIR / 'sample.jsonl').read_bytes()
        if hashlib.sha256(sample_bytes).hexdigest() != manifest['sample_sha256']:
            raise ValueError('Hash sample của run cũ đã thay đổi.')
        with (RUN_DIR / 'sample.jsonl').open(encoding='utf-8') as stream:
            sample = [json.loads(line) for line in stream if line.strip()]
        if {r['id']: r['url'] for r in sample} != {r['id']: r['url'] for r in selected} or len(sample) != len(selected):
            raise ValueError('ID/URL của run cũ không khớp khoảng chọn.')
        print('Manifest hợp lệ; giữ checkpoint để resume.')


## 6. Crawl — chạy lại ô này để tiếp tục

Tiến độ in mỗi 25 URL hoàn tất hoặc khi hết lượt. Chế độ robots theo `RESPECT_ROBOTS`,
giới hạn một request cùng registered domain, có retry và checkpoint theo ID.
`WORKERS = 4` không có nghĩa bốn request đồng thời đến cùng website.
ID đã có checkpoint, kể cả thất bại, sẽ được bỏ qua khi resume.
`MAX_NEW` giới hạn URL Requests mới; Selenium tiếp tục các URL lỗi, text rỗng hoặc
QC lỗi đã có checkpoint nhưng chưa thử browser. Chrome được kiểm tra trước lượt Requests;
nếu không khởi động được, ô crawl dừng và ghi lỗi riêng, giữ checkpoint hiện có.
Kết quả cuối ưu tiên engine có QC tốt hơn, giữ Requests khi hai kết quả tương đương.
Parser 120ask giữ câu hỏi/câu trả lời; cnkang giữ thân bài, bỏ menu và bài liên quan.
`records/` là kết quả cuối dùng để audit/export; `requests_records/` và
`browser_records/` giữ bằng chứng của từng engine. Chỉ nội dung qua QC mới được xuất corpus.
ZIP `<run>_diagnostics.zip` được cập nhật sau mỗi log tiến độ và khi lệnh kết thúc,
kể cả ngắt ô crawl. File có một số mẫu record/raw để debug, không nén lại toàn bộ raw.

Mặc định `BROWSER_TEST_MODE = 'failed_only'`: sau khi Requests hết lượt retry,
mọi lỗi HTTP/mạng/timeout/challenge, lỗi trích xuất, text rỗng và QC `review`/`rejected`
đều tự được đưa sang Selenium, kể cả PDF cần kiểm tra lại. Log `selenium_fallback_scheduled`
ghi số URL được chuyển và lý do. `records/` giữ `selenium_fallback_reason` cùng lỗi Requests gốc;
nội dung Selenium có QC tốt hơn được chọn và xuất vào corpus nếu đạt `accepted`.
Chrome/driver không khởi động được sẽ dừng sớm và lưu lỗi riêng. Selenium vẫn có thể thất bại.
`off` là lựa chọn chủ động tắt fallback; robots bị chặn khi `RESPECT_ROBOTS = True`
và URL không hợp lệ/địa chỉ không được phép vẫn không được chuyển sang browser.
Sau cập nhật snapshot, dùng `RUN_TAG` mới; có thể đặt `RECOVERY_RUN_PATH` để phục hồi run cũ.


In [ ]:
with diagnostic_step('crawl'):
    corpus_settings = {
        'user_agent': USER_AGENT, 'delay': DELAY, 'attempts': ATTEMPTS,
        'timeout': [PAGE_CONNECT_TIMEOUT, PAGE_READ_TIMEOUT], 'budget': PAGE_REQUEST_BUDGET,
        'respect_robots': RESPECT_ROBOTS,
        'robots_options': {'robots_attempts': ROBOTS_ATTEMPTS,
                           'robots_connect_timeout': ROBOTS_CONNECT_TIMEOUT,
                           'robots_read_timeout': ROBOTS_READ_TIMEOUT,
                           'robots_retry_budget': ROBOTS_RETRY_BUDGET,
                           'robots_failure_ttl': ROBOTS_FAILURE_TTL},
        'browser_mode': BROWSER_TEST_MODE, 'browser_user_agent': BROWSER_USER_AGENT,
        'chrome_binary': str(CHROME_BINARY) if CHROME_BINARY else None,
        'browser_load_timeout': BROWSER_LOAD_TIMEOUT, 'browser_wait_seconds': BROWSER_WAIT_SECONDS,
        'browser_attempts': BROWSER_ATTEMPTS,
    }
    settings_path = RUN_DIR / 'corpus_settings.json'
    settings_path.write_text(json.dumps(corpus_settings, ensure_ascii=False, indent=2), encoding='utf-8')
    arguments = ['crawl', '--out', RUN_DIR, '--workers', WORKERS, '--config', settings_path]
    if RECOVERY_RUN_PATH:
        arguments += ['--recovery-source', RECOVERY_RUN_PATH]
    if MAX_NEW is not None:
        arguments += ['--max-new', MAX_NEW]
    print('ZIP chẩn đoán tự cập nhật:', diagnostics_archive)
    run_script(CORPUS_RUNNER, *arguments)


## 7. Báo cáo, kiểm tra và xuất corpus

Corpus accepted chỉ gồm tài liệu qua heuristic; chưa chứng minh chất lượng bằng chấm thủ công.
Audit chỉ chạy khi đủ checkpoint. Nếu đang crawl thử, vẫn xuất phần accepted hiện có;
manifest notebook ghi rõ đây là corpus chưa hoàn tất.


In [ ]:
with diagnostic_step('report_audit_export'):
    run_script(CRAWLER, 'report', '--out', RUN_DIR)
    summary = json.loads((RUN_DIR / 'quality_report.json').read_text(encoding='utf-8'))
    display({k: summary[k] for k in ['selected', 'completed', 'pending', 'accepted', 'review', 'rejected', 'status_counts']})
    if (RUN_DIR / 'access_summary.json').exists():
        display(json.loads((RUN_DIR / 'access_summary.json').read_text(encoding='utf-8')))
        print('Lỗi theo tổng URL dataset giảm dần:', RUN_DIR / 'WEB_ERRORS.md')
        print('Đối chiếu Requests/Selenium:', RUN_DIR / 'requests_vs_selenium.csv')
    if summary['pending'] == 0:
        run_script(AUDITOR, '--out', RUN_DIR)
    else:
        print('Run chưa hoàn tất; bỏ qua audit toàn mẫu. Chạy lại ô crawl để tiếp tục.')
    if summary['accepted'] > 0:
        run_script(EXPORTER, '--out', RUN_DIR)
        corpus = pq.read_table(RUN_DIR / 'corpus.parquet')
        print(corpus.schema)
        display(corpus.select(['id', 'url', 'title']).slice(0, 5).to_pylist())
    else:
        print('Chưa có tài liệu accepted; xem quarantine.jsonl và QUALITY_REPORT.md.')


## 8. ZIP chẩn đoán — tích hợp trong Output

Chạy ô này để tạo lại gói debug từ checkpoint hiện có, kể cả khi crawl chưa xong.
Không cần file Python riêng, không tải lại URL. ZIP chứa:
- `overview.html`, `summary.json`: tiến độ, tốc độ/ETA, trạng thái HTTP và QC.
- CSV từng ID, lỗi/pending, request/retry và redirect.
- Log pip/crawl/Chrome, kết quả preflight, traceback notebook, manifest/cấu hình và phiên bản môi trường.
- Snapshot code, bằng chứng robots và tối đa 10 mẫu record/raw để kiểm tra parser.

Raw mẫu có ngân sách 10 MiB; các mẫu bị bỏ qua được ghi trong `sample_manifest.json`.
Giải nén ZIP và mở `overview.html`. Có thể gửi nguyên ZIP này để chẩn đoán lỗi.


In [ ]:
from IPython.display import FileLink
import zipfile

refresh_diagnostics(strict=True)
with zipfile.ZipFile(diagnostics_archive) as diagnostic_bundle:
    diagnostic_summary = json.loads(diagnostic_bundle.read('summary.json'))
display({key: diagnostic_summary[key] for key in ['selected', 'completed', 'pending', 'quality_status_counts', 'timing']})
print('ZIP chẩn đoán trong Output:', diagnostics_archive)
display(FileLink(str(diagnostics_archive.relative_to(WORKDIR))))


## 9. Đóng gói corpus đầy đủ để tải về

ZIP chứa cả raw, records, snapshot code và corpus để truy vết/resume; có thể khá lớn.
ZIP riêng `<run>_diagnostics.zip` nhỏ hơn: giải nén và mở `overview.html` để xem
tiến độ, tốc độ/ETA, HTTP status, lỗi và cờ chất lượng. Có CSV từng ID và log cài dependencies.
Tải file trong Output hoặc qua link bên dưới. Khi đổi session, cần giữ output hoặc
thêm run đã giải nén làm Kaggle Input và đặt `RESUME_RUN_PATH` trước khi chạy lại.
Resume không tự thử lại ID lỗi đã có checkpoint. Dùng `RECOVERY_RUN_PATH`
cùng `RUN_TAG` mới để thử lại lỗi bằng code/cấu hình mới và tái sử dụng raw thành công.


In [ ]:
from IPython.display import FileLink

notebook_manifest = {
    'id_from_inclusive': ID_FROM, 'id_to_inclusive': ID_TO,
    'dataset_revision': DATASET_REVISION, 'source_sha256': SOURCE_SHA256,
    'selected': summary['selected'], 'completed': summary['completed'],
    'pending': summary['pending'], 'accepted': summary['accepted'],
    'complete': summary['pending'] == 0,
    'packages': versions,
    'exported_files': [name for name in ['corpus.parquet', 'corpus.jsonl'] if (RUN_DIR / name).exists()],
}
(RUN_DIR / 'notebook_manifest.json').write_text(json.dumps(notebook_manifest, ensure_ascii=False, indent=2), encoding='utf-8')
refresh_diagnostics(strict=True)
archive = shutil.make_archive(str(WORKDIR / RUN_DIR.name), 'zip', root_dir=RUN_DIR.parent, base_dir=RUN_DIR.name)
print('ZIP:', archive)
print('Corpus:', RUN_DIR / 'corpus.parquet')
display(FileLink(str(Path(archive).relative_to(WORKDIR))))
print('ZIP chẩn đoán:', diagnostics_archive)
display(FileLink(str(diagnostics_archive.relative_to(WORKDIR))))
if (RUN_DIR / 'corpus.parquet').exists():
    display(FileLink(str((RUN_DIR / 'corpus.parquet').relative_to(WORKDIR))))
    display(FileLink(str((RUN_DIR / 'corpus.jsonl').relative_to(WORKDIR))))
